# RAG pour la sécurité des machines industrielles
**Question-réponse sur une collection documentaire spécialisée**
Auteur : Mohamed Ghandour

> **Note de transparence.** Ce projet a été réalisé avec l'assistance de **Claude Code** (Anthropic) pour le code des modules `src/`, les brouillons de questions du jeu de test et une première version de ce notebook. Les annotations utilisées pour valider le juge LLM ont été produites par Claude, non par un humain (voir §6). Seules 20 questions du jeu de test ont été relues à la main (voir §7.5) : les résultats fins sont **préliminaires** (voir §9).

## 1. Introduction et problématique

Un grand modèle de langage (LLM) répond de mémoire. Dans un domaine réglementaire et technique comme la **sécurité des machines**, cela pose deux problèmes : il peut **inventer** une valeur, un seuil ou un article de loi qui paraît plausible, et il ne peut pas **citer sa source**. Une erreur sur une distance de sécurité ou une fréquence de vérification n'est pas anodine.

**Le RAG (Retrieval-Augmented Generation)** répond à ce problème en deux temps : (1) *retrouver* dans un corpus les passages pertinents, (2) *générer* la réponse en s'appuyant uniquement sur ces passages, avec citation des sources et possibilité de s'abstenir.

**Problématique.** *Sur un corpus de documents français sur la sécurité des machines (brochures INRS, règlement UE 2023/1230, directive 2006/42/CE), quelle chaîne de recherche et de génération donne les réponses les plus exactes, les mieux sourcées et les moins hallucinées, et à quel coût ?*

Quatre systèmes sont définis : **S0** (LLM seul, baseline), **S1** (BM25 + LLM), **S2** (embeddings denses + LLM), **S3** (hybride + reranker + LLM). Les quatre sont comparés en **retrieval** (132 questions) et en **génération** (34 questions, quota d'API limité) ; S0 et S1 le sont aussi sur 61 questions. Le sujet a aussi un lien direct avec mon stage en ML industriel chez SAMM Automation.

## 2. Bref état de l'art

- **RAG** : Lewis et al. (2020) proposent d'augmenter un générateur par un module de recherche pour réduire les hallucinations et rendre les réponses traçables.
- **Recherche lexicale, BM25** : Robertson & Zaragoza (2009). Un classique robuste, souvent difficile à battre sur des termes techniques.
- **Recherche dense** : des encodeurs transforment textes et requêtes en vecteurs ; ici **bge-m3** (Chen et al., 2024), multilingue et à contexte long.
- **Hybride et fusion** : combiner lexical et dense par *Reciprocal Rank Fusion* (Cormack et al., 2009).
- **Reranking** : un *cross-encoder* relit chaque couple (question, passage) ; ici **bge-reranker-v2-m3**.
- **Évaluation par LLM-juge** : Zheng et al. (2023) (*MT-Bench / LLM-as-a-judge*), avec des biais connus (préférence pour ses propres sorties, pour les réponses longues).

## 3. Données et préparation

**Corpus** : 40 documents, 1 516 pages — 38 brochures INRS et 2 textes de l'UE (règlement 2023/1230, directive 2006/42/CE). Sources dans `data/sources.csv`. Les PDF ne sont pas versionnés (`data/raw/`).

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (ROOT / "data" / "sources.csv").exists():     # exécution hors du dépôt (ex. Colab) : récupérer le projet
    import subprocess
    subprocess.run(["git", "clone", "-q", "https://github.com/Mohamed-Ghandour/rag-securite-machines.git", "projet"], check=True)
    ROOT = Path("projet").resolve()
sys.path.insert(0, str(ROOT))
pd.set_option("display.width", 160, "display.max_colwidth", 90)

src = pd.read_csv(ROOT / "data/sources.csv")
print(len(src), "documents,", src.pages.sum(), "pages")
src.assign(organisme=src.fichier.str.startswith("EURLEX").map({True: "EUR-Lex", False: "INRS"})).groupby("organisme").agg(documents=("fichier", "count"), pages=("pages", "sum"))

### 3.1 Extraction du texte

J'ai comparé deux bibliothèques sur des pages réelles : **pymupdf** (rapide, bon ordre de lecture, casse des sigles conservée) et **pdfplumber** (meilleur sur les tableaux). Aucune ne gagne partout, d'où une **extraction hybride** (`src/extract.py`) :
- texte courant : pymupdf ; tableaux : pdfplumber, réécrits « colonne : valeur » pour que chaque ligne garde ses en-têtes ;
- titres repérés par la taille de police (INRS) ou par le motif « Article / CHAPITRE / ANNEXE » (EUR-Lex) ;
- nettoyage : en-têtes et pieds de page répétés, césures résolues avec un vocabulaire du corpus (« at-traction » → « attraction », mais « main-bras » conservé), puces Wingdings, caractères parasites, cases à cocher.

**Limite** : les figures et pictogrammes ne sont pas extraits, seulement leurs légendes.

In [ ]:
try:
    d = json.load(open(ROOT / "data/processed/text/INRS_ED828.json", encoding="utf-8"))
    print(d["pages"][13]["text"][:900])
except FileNotFoundError:
    print("Cache d'extraction absent : exécuter d'abord  python -m src.download_corpus  puis  python -m src.extract")

### 3.2 Découpage en chunks (*chunking*)

On ne peut pas donner 1 500 pages à un LLM ni chercher efficacement dans de gros blocs : on découpe en **chunks**. Deux stratégies sont comparées, sur le **même texte nettoyé** :
- **fixe** : fenêtre glissante de N tokens avec 15 % de recouvrement, sans égard à la structure (baseline) ;
- **structurel** : on ne traverse jamais un titre ; les paragraphes sont regroupés jusqu'à N tokens ; une ligne de tableau n'est jamais coupée ; chaque chunk reçoit un fil d'Ariane (« Règlement 2023/1230 > CHAPITRE V > Article 32 »).

Trois tailles : 256, 512, 1024 tokens (comptés avec le tokenizer de bge-m3). Le contexte est compté **dans** le budget pour que la comparaison soit équitable.

In [ ]:
try:
    rows = []
    for strat in ("fixed", "structural"):
        for size in (256, 512, 1024):
            ch = [json.loads(l) for l in open(ROOT / f"data/processed/chunks/{strat}_{size}.jsonl", encoding="utf-8")]
            t = pd.Series([c["n_tokens"] for c in ch])
            rows.append({"stratégie": strat, "taille": size, "chunks": len(ch), "tokens moyens": round(t.mean()), "max": t.max()})
    display(pd.DataFrame(rows))
    ch = [json.loads(l) for l in open(ROOT / "data/processed/chunks/structural_512.jsonl", encoding="utf-8")]
    c = next(c for c in ch if c["doc"].startswith("EURLEX_reglement") and "Article 32" in c["section"])
    print("[contexte]", c["context"]); print(c["text"][:500])
except FileNotFoundError:
    print("Chunks absents : exécuter  python -m src.chunking  (après python -m src.extract)")

### 3.3 Jeu de test

Des **brouillons de questions** ont été générés avec un LLM (`gpt-oss-120b`, puis `qwen3.8-27b` quand le quota du premier a été atteint) à partir de passages tirés du corpus, en quatre types : **factuelle**, **numérique/tabulaire**, **multi-passages** (deux sources à combiner) et **sans réponse** dans le corpus (pour tester l'abstention).

Garde-fous automatiques : le LLM doit recopier **mot pour mot** la phrase qui prouve la réponse, et le code vérifie qu'elle existe bien dans le document (63 appels sur 230 rejetés : preuve introuvable, question non autonome…) ; les doublons sont retirés. Résultat : **167 questions candidates** (74 factuelles, 37 numériques, 21 multi-passages, 35 sans réponse), couvrant 36 documents.

Un fichier de relecture (`questions_a_verifier.csv`) permet de valider chaque question à la main : **20 questions factuelles ont été relues et validées** (§7.5) ; les autres restent à relire. Voir les limites en §9.

In [ ]:
qs = [json.loads(l) for l in open(ROOT / "data/testset/brouillon.jsonl", encoding="utf-8")]
print(pd.Series([q["type"] for q in qs]).value_counts().to_string())
q = next(q for q in qs if q["type"] == "numerique"); print("\nExemple :", q["question"], "\nRéférence :", q["reponse"], "\nPreuve :", q["preuves"][0]["quote"][:150])

## 4. Méthodologie

### 4.1 Notions utilisées (expliquées avant usage)

**BM25 (recherche lexicale).** Il note un chunk selon les mots de la question qu'il contient : un mot rare compte plus qu'un mot courant (*idf*), les occurrences répétées sont saturées (*k1*), et la longueur du chunk est normalisée (*b*). Excellent pour les termes exacts (sigles, normes, valeurs), aveugle aux synonymes. Implémenté à la main (`src/bm25.py`) et vérifié contre la bibliothèque `rank_bm25`.

**Embeddings (recherche dense).** Un modèle (bge-m3) transforme chaque texte en un vecteur de 1024 nombres ; deux textes de sens proche ont des vecteurs proches (similarité cosinus). Cela capture reformulations et synonymes, mais est plus flou sur les valeurs exactes. FAISS cherche rapidement les vecteurs les plus proches.

**Fusion RRF (*Reciprocal Rank Fusion*).** On fusionne deux classements sans comparer leurs scores (d'échelles différentes) : chaque chunk reçoit la somme de 1/(60 + rang) sur les classements. Un chunk bien classé par les deux méthodes remonte.

**Reranker cross-encoder.** Les embeddings encodent la question et le chunk *séparément*. Un cross-encoder lit le couple *ensemble* et prédit sa pertinence : plus précis, mais plus lent, donc appliqué seulement aux 30 meilleurs candidats.

**LLM-juge.** Un second LLM note les réponses (correction par rapport à la référence ; fidélité aux extraits). Rapide, mais biaisé : on **valide le juge** en mesurant son accord avec des annotations (**kappa de Cohen**, accord corrigé du hasard : 0 = hasard, 1 = accord parfait).

**Bootstrap.** On rééchantillonne les questions avec remise (5 000 fois) pour obtenir un **intervalle de confiance à 95 %** sur chaque métrique et sur la **différence entre deux systèmes**, comparés sur les mêmes questions. Si l'intervalle d'une différence exclut 0, elle est jugée significative.

### 4.2 Systèmes comparés
| # | Système | Rôle |
|---|---|---|
| S0 | LLM seul (closed-book) | Baseline : que sait le modèle sans documents ? |
| S1 | BM25 + LLM | Baseline lexicale |
| S2 | bge-m3 + FAISS + LLM | RAG standard |
| S3 | BM25 + dense, fusion RRF, reranker, + LLM | Système avancé |

Générateur : `openai/gpt-oss-20b` (petit, rapide, open-weights, via l'API Groq). Juge : `qwen/qwen3.8-27b`, d'une **autre famille** que le générateur pour limiter le biais d'auto-préférence. Prompt RAG : réponse concise, **citations obligatoires** `[n]`, et formule d'abstention imposée si les extraits ne contiennent pas la réponse. Température 0.

### 4.3 Justification des choix de modèles et de paramètres

| Brique | Choix | Pourquoi | Ce qui n'a pas été comparé |
|---|---|---|---|
| Générateur | `gpt-oss-20b` | Modèle **open-weights** (reproductible, pas de dépendance à un fournisseur propriétaire), d'environ 20 milliards de paramètres, rapide (latence sous la seconde) et accessible gratuitement via Groq. Un **petit** modèle rend l'expérience plus exigeante : s'il s'abstient et cite correctement avec de bons extraits, c'est que le retrieval fait le travail. | Un modèle plus grand (`gpt-oss-120b`, disponible) n'a pas été comparé. Le choix est aussi **contraint par le quota gratuit**. |
| Juge | `qwen3.8-27b` | **Famille différente** du générateur, pour limiter le biais d'auto-préférence des juges LLM ; température 0 et sortie JSON pour des notes reproductibles. | Pas de comparaison entre plusieurs juges ; accord avec un humain non mesuré (§6). |
| Embeddings | `bge-m3` | **Multilingue** (le corpus est en français), contexte long (jusqu'à 8 192 tokens : nos chunks de 1 024 tokens ne sont pas tronqués), 2,6 Go de VRAM en demi-précision, donc compatible avec un GPU de 6 Go. | Pas de comparaison avec d'autres encodeurs (e5, etc.) ni de fine-tuning. |
| Reranker | `bge-reranker-v2-m3` | Cross-encoder **multilingue** de la même famille que bge-m3, qui tient aussi en mémoire. | Pas de comparaison avec d'autres rerankers. |
| BM25 | k1 = 1,5 ; b = 0,75 | Valeurs usuelles de la littérature. | Pas de réglage sur les données. |
| Taille des chunks | 256, 512, 1024 tokens | Compromis classique : petit = précis mais contexte pauvre, grand = contexte riche mais bruité. Tranché **expérimentalement** (§7.2). | |
| Nombre d'extraits (k) | 5 | Compromis entre couverture (Recall@5 élevé) et coût en tokens (~2 100 tokens par requête). | Pas d'ablation de k en génération. |

## 5. Implémentation

Le pipeline est codé à la main dans `src/` (pas de LangChain / LlamaIndex) : `extract.py`, `chunking.py`, `bm25.py`, `dense.py`, `hybrid.py`, `retrieval_eval.py`, `rag.py`, `analyze.py`. Tout ce qui coûte cher est **mis en cache** (texte extrait, embeddings, réponses du LLM), les graines sont fixées et les résultats sont écrits dans `results/`. Ce notebook **relit ces résultats** et n'appelle aucune API.

### Code source intégré
Pour que ce notebook contienne tout le code, **les modules sont reproduits ci-dessous** : chaque cellule `%%writefile` écrit le module dans `src/` (à exécuter en premier sur une machine neuve, par exemple Colab). Les autres cellules les importent ensuite.

**`src/extract.py`** — Extraction du texte des PDF (pymupdf + pdfplumber, tableaux, titres, nettoyage)

In [ ]:
%%writefile src/extract.py
"""Extraction du texte des PDF (étape 1 du pipeline RAG).

Approche hybride, choisie après comparaison sur des pages réelles du corpus :
  - texte courant : pymupdf (rapide, ordre de lecture correct, casse des sigles
    préservée) ;
  - tableaux      : pdfplumber (texte propre, cellules fusionnées bien gérées),
    réécrits ligne par ligne « colonne : valeur » pour que chaque ligne garde
    ses en-têtes une fois découpée en chunks ;
  - titres        : repérés par la taille de police (INRS) ou par la forme
    « Article N », « CHAPITRE », « ANNEXE » (textes EUR-Lex) ;
  - nettoyage     : en-têtes/pieds de page répétés, césures, puces Wingdings,
    numéros de paragraphe (« 1. ») recollés à leur texte.

Sortie : data/processed/text/<document>.json (cache), un objet par document :
{"doc": ..., "pages": [{"page": 1, "text": "...",
                        "blocks": [{"k": "p"|"h"|"tbl", "t": "...", "lvl": 2}, ...]}]}
(« lvl » n'existe que pour les titres, « k » vaut p = paragraphe, h = titre, tbl = tableau.)

Usage : python -m src.extract
"""
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path

import pdfplumber
import pymupdf

ROOT = Path(__file__).resolve().parents[1]
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed" / "text"

MARGIN = 0.08          # fraction de la hauteur de page considérée comme marge (en-tête/pied)
MIN_REPEAT = 0.20      # une ligne de marge présente sur >= 20 % des pages est un en-tête/pied
HEADING_RATIO = 1.3    # un titre INRS a une police >= 1,3 x celle du corps de texte
BULLET_FONTS = ("wingdings", "dingbats", "symbol")
LOWER = "a-zàâçéèêëîïôûùüœ"

# Caractères spéciaux désignés par leur code (pas de caractères invisibles dans le source)
SOFT_HYPHEN = chr(0x00AD)
HYPH = chr(0x2010)     # marqueur « coupure de ligne ambiguë », résolu par resolve_hyphens()
LIGATURES = {chr(0xFB01): "fi", chr(0xFB02): "fl", chr(0xFB00): "ff", chr(0xFB03): "ffi", chr(0xFB04): "ffl"}
LIGATURE_CLASS = "[" + "".join(LIGATURES) + "]"
# « � », caractères de contrôle, zone d'usage privé (codes de puces Wingdings, etc.)
JUNK_RE = re.compile(f"[{chr(0xFFFD)}{chr(0)}-{chr(8)}{chr(0x0B)}-{chr(0x1F)}{chr(0xE000)}-{chr(0xF8FF)}]")
NARROW_SPACES = {0xA0: " ", 0x2009: " ", 0x202F: " "}

# Titres des textes EUR-Lex : motif -> niveau hiérarchique
EURLEX_HEADINGS = [
    (re.compile(r"(CHAPITRE|ANNEXE|TITRE)\s+[IVXLC\d]+\w*"), 1),
    (re.compile(r"(SECTION|PARTIE)\s+[\w\d]+"), 2),
    (re.compile(r"Article\s+(\d+\w?|premier)"), 3),
]


# ---------------------------------------------------------------- nettoyage du texte

def clean_text(text: str) -> str:
    """Caractères parasites, espaces spéciales, ligatures."""
    text = JUNK_RE.sub("", text).translate(NARROW_SPACES)
    text = re.sub(f"({LIGATURE_CLASS}) (?=[{LOWER}])", r"\1", text)   # « vérifi cateur » -> « vérificateur »
    text = text.translate(str.maketrans(LIGATURES))
    return re.sub(r"[ \t]{2,}", " ", text).strip()


def _block_text(block: dict) -> str:
    """Texte d'un bloc pymupdf (dict) : puces Wingdings -> '•', lignes recollées."""
    lines = []
    for line in block["lines"]:
        parts = []
        for span in line["spans"]:
            txt = span["text"]
            if any(f in span["font"].lower() for f in BULLET_FONTS) and len(txt.strip()) == 1:
                txt = "•"
            parts.append(txt)
        lines.append("".join(parts).strip())
    text = "\n".join(l for l in lines if l)
    text = text.replace(SOFT_HYPHEN + "\n", "").replace(SOFT_HYPHEN, "")  # fin de ligne : on recolle
    # « per-\nmettre » : coupure ambiguë (césure ou vrai trait d'union ?) -> marqueur, résolu plus tard
    text = re.sub(r"(\w)-\n(?=[" + LOWER + "])", r"\1" + HYPH, text)
    return clean_text(re.sub(r"\s*\n\s*", " ", text))


def _in_table(bbox, tables) -> bool:
    """Vrai si le centre du bloc tombe dans l'un des tableaux de la page."""
    cx, cy = (bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2
    return any(t[0] <= cx <= t[2] and t[1] <= cy <= t[3] for t in tables)


# ---------------------------------------------------------------- tableaux

def _clean_cell(c) -> str:
    c = (c or "").replace(SOFT_HYPHEN + "\n", "").replace(SOFT_HYPHEN, "")
    c = re.sub(r"\(cid:\d+\)", chr(0x2610), c)   # glyphe de case à cocher non décodé par pdfplumber (☐)
    return clean_text(re.sub(r"\s+", " ", c))


def _render_table(rows: list[list]) -> str:
    """Tableau -> lignes « en-tête : valeur | en-tête : valeur » (lignes de section isolées)."""
    rows = [[_clean_cell(c) for c in r] for r in rows]
    header, body = rows[0], rows[1:]
    out = []
    for r in body:
        filled = [(h, v) for h, v in zip(header, r) if v]
        if not filled:
            continue
        if len(r) > 2 and len(filled) == 1 and r[0]:
            out.append(r[0])                               # ligne de section (cellule fusionnée)
        else:
            out.append(" | ".join(f"{h} : {v}" if h else v for h, v in filled))
    return "\n".join(out)


def _page_tables(plumber_page) -> list[tuple[tuple, str]]:
    """Vrais tableaux de la page : [(bbox, texte)]. Écarte les faux positifs
    (figures encadrées, schémas) via la proportion de cellules non vides."""
    found = []
    try:
        tables = plumber_page.find_tables()
    except Exception:
        return found
    for t in tables:
        rows = t.extract()
        if len(rows) < 2 or max(len(r) for r in rows) < 2:
            continue
        cells = [c for r in rows for c in r]
        nonempty = sum(bool(_clean_cell(c)) for c in cells)
        if nonempty / len(cells) < 0.35 or nonempty < 4:
            continue
        text = _render_table(rows)
        if text:
            found.append((t.bbox, text))
    return found


# ---------------------------------------------------------------- titres

def _heading_level(text: str, size: float, body: float, eurlex: bool):
    """Niveau du titre (1 = le plus haut) ou None si le bloc n'est pas un titre."""
    if eurlex:
        for pat, lvl in EURLEX_HEADINGS:
            if pat.fullmatch(text):
                return lvl
        return None
    if size < max(HEADING_RATIO * body, body + 3) or not 3 <= len(text) <= 150:
        return None
    if re.fullmatch(r"(\w ?)+", text) and len(text) < 12:      # lettrines « C H A P I T R E »
        return None
    if text.startswith("•") or len(re.findall(r"[^\W\d_]", text)) < 3:   # puces, symboles
        return None
    m = re.match(r"(\d+(?:\.\d+)*)\.?\s", text)
    if m:
        return m.group(1).count(".") + 1                       # « 4.3.1. » -> niveau 3
    return 1 if size >= 1.8 * body else 2


def _mark_headings(items: list[dict], body: float, eurlex: bool) -> None:
    """Étiquette les blocs « h » et fusionne les titres coupés sur plusieurs blocs
    (« 4.4. » + « Écartements… » + « d'écrasement » ; « Article 31 » + son intitulé)."""
    prev = None
    for it in items:
        if it["k"] != "p":
            prev = None
            continue
        lvl = None if it["margin"] else _heading_level(it["t"], it["size"], body, eurlex)
        if lvl is not None:
            it["k"], it["lvl"] = "h", lvl
        if eurlex:
            # l'intitulé d'un article/chapitre suit son numéro, avec le même corps de police
            if prev and prev["k"] == "h" and it["k"] == "p" and len(it["t"]) <= 150 and not prev.get("titled"):
                prev["t"] += " - " + it["t"]
                prev["titled"] = True
                it["k"] = "drop"
                continue
        elif prev and prev["k"] == "h" and it["k"] == "h" and abs(prev["size"] - it["size"]) < 0.5:
            starts_numbered = re.match(r"\d+(\.\d+)*\.?\s", it["t"])
            if re.fullmatch(r"\d+(\.\d+)*\.?", prev["t"]) or not starts_numbered:
                prev["t"] += " " + it["t"]
                it["k"] = "drop"
                continue
        prev = it


# ---------------------------------------------------------------- document

def _signature(text: str) -> str:
    """Forme normalisée d'une ligne de marge (chiffres masqués) pour repérer les répétitions."""
    t = unicodedata.normalize("NFKC", text).lower()
    return re.sub(r"\d+", "#", re.sub(r"\s+", " ", t)).strip()


def extract_document(path: Path) -> list[dict]:
    """Pages du PDF sous forme de blocs étiquetés (les coupures ambiguës portent le marqueur HYPH)."""
    eurlex = path.name.startswith("EURLEX_")
    doc = pymupdf.open(path)
    pages, size_chars = [], Counter()

    with pdfplumber.open(path) as pl:
        for i, page in enumerate(doc):
            H = page.rect.height
            tables = _page_tables(pl.pages[i])
            items = []
            for b in page.get_text("dict")["blocks"]:
                if b["type"] != 0 or _in_table(b["bbox"], [t[0] for t in tables]):
                    continue
                txt = _block_text(b)
                if not txt:
                    continue
                spans = [s for l in b["lines"] for s in l["spans"] if s["text"].strip()]
                size = max((s["size"] for s in spans), default=0)
                is_margin = b["bbox"][3] < MARGIN * H or b["bbox"][1] > (1 - MARGIN) * H
                if not is_margin:
                    for s in spans:
                        size_chars[round(s["size"] * 2) / 2] += len(s["text"])
                items.append({"y": b["bbox"][1], "t": txt, "k": "p", "margin": is_margin, "size": size})
            for bbox, txt in tables:
                items.append({"y": bbox[1], "t": txt, "k": "tbl", "margin": False, "size": 0})
            pages.append(items)

    body = size_chars.most_common(1)[0][0] if size_chars else 10.0

    # signatures de marge -> en-têtes/pieds répétés
    sig_pages = Counter()
    for items in pages:
        for s in {_signature(it["t"]) for it in items if it["margin"]}:
            sig_pages[s] += 1
    repeated = {s for s, n in sig_pages.items() if n >= max(2, int(MIN_REPEAT * len(pages)))}

    result = []
    for i, items in enumerate(pages, start=1):
        # ordre : blocs de texte dans l'ordre pymupdf ; chaque tableau est inséré
        # avant le premier bloc de texte situé sous son bord supérieur
        seq = [it for it in items if it["k"] != "tbl"]
        for t in (it for it in items if it["k"] == "tbl"):
            seq.insert(next((k for k, it in enumerate(seq) if it["y"] > t["y"]), len(seq)), t)
        seq = [it for it in seq if not (it["margin"] and (
            _signature(it["t"]) in repeated or re.fullmatch(r"\d{1,3}\s*\w?", it["t"])))]

        # numéros de paragraphe « 1. » / « (a) » isolés -> recollés au bloc suivant
        merged, pending = [], ""
        for it in seq:
            if it["k"] == "p" and re.fullmatch(r"(\d+\.|\(\w{1,3}\))", it["t"]):
                pending = it["t"] + " "
                continue
            it["t"] = pending + it["t"]
            pending = ""
            # coupure de mot à cheval sur deux blocs : « at- » + « traction »
            if (merged and it["k"] == "p" and merged[-1]["k"] == "p"
                    and re.search(r"\w-$", merged[-1]["t"]) and re.match(f"[{LOWER}]", it["t"])):
                merged[-1]["t"] = merged[-1]["t"][:-1] + HYPH + it["t"]
            else:
                merged.append(it)

        _mark_headings(merged, body, eurlex)
        blocks = []
        for it in merged:
            if it["k"] == "drop":
                continue
            blocks.append({"k": it["k"], "t": it["t"], **({"lvl": it["lvl"]} if it["k"] == "h" else {})})
        result.append({"page": i, "blocks": blocks})
    return result


def resolve_hyphens(texts: list[str], vocab_texts: list[str]) -> list[str]:
    """Résout les coupures ambiguës « mot‐suite » avec un vocabulaire tiré du corpus :
    on recolle (« at‐traction » -> « attraction ») si la forme soudée existe ailleurs
    dans le corpus, sinon on garde le trait d'union (« main‐bras » -> « main-bras »)."""
    word = r"[^\W\d_]+"
    vocab = Counter(w.lower() for t in vocab_texts for w in re.findall(word, t.replace(HYPH, " ")))

    def repl(m):
        a, b = m.group(1), m.group(2)
        return a + b if vocab[(a + b).lower()] > 0 else f"{a}-{b}"

    out = [re.sub(f"({word}){HYPH}({word})", repl, t) for t in texts]
    return [t.replace(HYPH, "-") for t in out]       # marqueurs restants (chiffres, etc.)


def _page_text(blocks: list[dict]) -> str:
    return "\n".join(b["t"] for b in blocks)


def main() -> None:
    OUT.mkdir(parents=True, exist_ok=True)
    fresh = {}                                           # documents extraits lors de cet appel
    for pdf in sorted(RAW.glob("*.pdf")):
        dest = OUT / f"{pdf.stem}.json"
        if dest.exists() and dest.stat().st_mtime >= pdf.stat().st_mtime:
            continue                                     # cache valide
        fresh[pdf.stem] = extract_document(pdf)
    if not fresh:
        return

    # vocabulaire du corpus complet (documents en cache + nouveaux) pour résoudre les césures
    vocab_texts = [b["t"] for pages in fresh.values() for p in pages for b in p["blocks"]]
    for f in OUT.glob("*.json"):
        if f.stem not in fresh:
            for p in json.loads(f.read_text(encoding="utf-8"))["pages"]:
                vocab_texts += [b["t"] for b in p["blocks"]]

    for name, pages in fresh.items():
        blocks = [b for p in pages for b in p["blocks"]]
        for b, t in zip(blocks, resolve_hyphens([b["t"] for b in blocks], vocab_texts)):
            b["t"] = t
        for p in pages:
            p["text"] = _page_text(p["blocks"])
        (OUT / f"{name}.json").write_text(json.dumps({"doc": name, "pages": pages}, ensure_ascii=False), encoding="utf-8")
        n_head = sum(b["k"] == "h" for b in blocks)
        print(f"{name:42} {len(pages):4} p. {sum(len(p['text']) for p in pages):8} car. {n_head:4} titres")


if __name__ == "__main__":
    main()

**`src/chunking.py`** — Découpage fixe et structurel

In [ ]:
%%writefile src/chunking.py
"""Découpage du corpus en chunks (étape 2 du pipeline RAG).

Deux stratégies comparées en ablation, appliquées sur la MÊME entrée nettoyée :
  - fixe       : fenêtre glissante de N tokens avec recouvrement, sans tenir compte
                 de la structure du document (baseline) ;
  - structurel : on ne traverse jamais un titre (article, section) ; les paragraphes
                 sont regroupés jusqu'à N tokens et les lignes de tableau ne sont
                 jamais coupées au milieu.

Chaque chunk porte un « contexte » (titre du document, et pour la stratégie
structurelle le fil d'Ariane des titres) ajouté devant le texte pour l'indexation.
Le contexte est compté dans le budget de N tokens : un chunk indexé ne dépasse
jamais N tokens, ce qui rend les tailles comparables.

Les tokens sont comptés avec le tokenizer de bge-m3, le modèle d'embeddings
envisagé : « 512 tokens » est donc ce que le modèle verra réellement.

Usage : python -m src.chunking
"""
import csv
import json
import re
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path

from transformers import AutoTokenizer

ROOT = Path(__file__).resolve().parents[1]
TEXT_DIR = ROOT / "data" / "processed" / "text"
CHUNK_DIR = ROOT / "data" / "processed" / "chunks"
SOURCES_CSV = ROOT / "data" / "sources.csv"

TOKENIZER_NAME = "BAAI/bge-m3"
MIN_TOKENS = 80          # (structurel) une section plus courte est fusionnée avec la suivante
OVERLAP = 0.15           # (fixe) recouvrement entre fenêtres consécutives
BOILERPLATE_MIN_DOCS = 3  # un bloc identique dans >= 3 documents est du texte « maison » répété


@lru_cache(maxsize=1)
def tokenizer():
    return AutoTokenizer.from_pretrained(TOKENIZER_NAME)


def n_tokens(text: str) -> int:
    return len(tokenizer()(text, add_special_tokens=False)["input_ids"])


# ---------------------------------------------------------------- chargement + nettoyage commun

def _norm(t: str) -> str:
    return re.sub(r"\s+", " ", t.lower()).strip()


def _is_toc_page(blocks: list[dict]) -> bool:
    """Page de sommaire : surtout des lignes « 1.2. Titre 14 » (numéro de page en fin de ligne)."""
    lines = [l for b in blocks for l in b["t"].split("\n") if l.strip()]
    if any(_norm(b["t"]) in ("sommaire", "table des matières") for b in blocks):
        return True
    toc = sum(bool(re.search(r"\S\s+\d{1,3}$", l)) and bool(re.match(r"\d", l)) for l in lines)
    return len(lines) >= 6 and toc >= 0.5 * len(lines)


def load_corpus() -> dict[str, dict]:
    """{doc: {"title": ..., "blocks": [{"page", "k", "t", "lvl"?}]}} nettoyé :
    sommaires retirés, blocs « maison » répétés dans plusieurs documents retirés."""
    titles = {}
    if SOURCES_CSV.exists():
        with open(SOURCES_CSV, encoding="utf-8") as f:
            titles = {Path(r["fichier"]).stem: r["titre"] for r in csv.DictReader(f)}

    docs, stats = {}, Counter()
    for f in sorted(TEXT_DIR.glob("*.json")):
        d = json.loads(f.read_text(encoding="utf-8"))
        blocks = []
        for p in d["pages"]:
            if _is_toc_page(p["blocks"]):
                stats["pages_sommaire"] += 1
                continue
            blocks += [{**b, "page": p["page"]} for b in p["blocks"]]
        docs[d["doc"]] = {"title": titles.get(d["doc"], d["doc"]), "blocks": blocks}

    # texte « maison » (présentation de l'INRS, mentions légales...) répété dans plusieurs documents
    seen = defaultdict(set)
    for name, d in docs.items():
        for b in d["blocks"]:
            if b["k"] == "p" and len(b["t"]) >= 40:
                seen[_norm(b["t"])].add(name)
    boiler = {t for t, ds in seen.items() if len(ds) >= BOILERPLATE_MIN_DOCS}
    for d in docs.values():
        kept = [b for b in d["blocks"] if not (b["k"] == "p" and _norm(b["t"]) in boiler)]
        stats["blocs_maison"] += len(d["blocks"]) - len(kept)
        d["blocks"] = kept
    load_corpus.stats = dict(stats)
    return docs


# ---------------------------------------------------------------- stratégie 1 : fixe

def chunk_fixed(doc: str, d: dict, size: int, overlap: float = OVERLAP) -> list[dict]:
    """Fenêtres de `size` tokens (contexte inclus), recouvrement `overlap`, sans égard à la structure."""
    context = d["title"]
    budget = size - n_tokens(context) - 4      # marge : le texte découpé est re-tokenisé avec son contexte
    text, spans = "", []                       # spans : (début, fin, page) de chaque bloc dans `text`
    for b in d["blocks"]:
        start = len(text)
        text += b["t"] + "\n"
        spans.append((start, len(text), b["page"]))
    if not text.strip():
        return []

    enc = tokenizer()(text, add_special_tokens=False, return_offsets_mapping=True)
    offs = enc["offset_mapping"]
    step = max(1, int(budget * (1 - overlap)))
    chunks, i = [], 0
    while i < len(offs):
        j = min(i + budget, len(offs))
        c0, c1 = offs[i][0], offs[j - 1][1]
        chunks.append(_make(doc, d, context, text[c0:c1].strip(), _pages(spans, c0, c1), ""))
        if j == len(offs):
            break
        i += step
    return chunks


def _pages(spans, c0, c1):
    pages = [p for s, e, p in spans if e > c0 and s < c1]
    return (pages[0], pages[-1]) if pages else (spans[-1][2], spans[-1][2])


# ---------------------------------------------------------------- stratégie 2 : structurelle

def _units(block: dict, budget: int) -> list[str]:
    """Unités insécables d'un bloc : le paragraphe, ou chaque ligne d'un tableau.
    Une unité trop longue est coupée par phrases, puis en dernier recours par tokens."""
    raw = block["t"].split("\n") if block["k"] == "tbl" else [block["t"]]
    out = []
    for u in raw:
        if n_tokens(u) <= budget:
            out.append(u)
            continue
        for s in re.split(r"(?<=[.;:!?])\s+", u):
            if n_tokens(s) <= budget:
                out.append(s)
            else:                                     # phrase géante : découpe dure par tokens
                enc = tokenizer()(s, add_special_tokens=False, return_offsets_mapping=True)
                o = enc["offset_mapping"]
                out += [s[o[k][0]:o[min(k + budget, len(o)) - 1][1]] for k in range(0, len(o), budget)]
    return out


def chunk_structural(doc: str, d: dict, size: int) -> list[dict]:
    chunks, stack = [], {}              # stack : niveau -> texte du titre courant
    cur, cur_tok, cur_pages, cur_ctx = [], 0, [], ""

    def crumb():
        return " > ".join([d["title"]] + [stack[k] for k in sorted(stack)])

    def flush():
        nonlocal cur, cur_tok, cur_pages
        if cur:
            chunks.append(_make(doc, d, cur_ctx, "\n".join(cur), (cur_pages[0], cur_pages[-1]), cur_ctx))
        cur, cur_tok, cur_pages = [], 0, []

    for b in d["blocks"]:
        if b["k"] == "h":
            if cur_tok >= MIN_TOKENS:
                flush()                                   # on ne traverse pas un titre (sauf section minuscule)
            stack = {k: v for k, v in stack.items() if k < b["lvl"]}
            stack[b["lvl"]] = b["t"]
            if not cur:
                cur_ctx = crumb()
            continue
        if not cur:
            cur_ctx = crumb()
        budget = size - n_tokens(cur_ctx) - 1
        for u in _units(b, max(budget, 32)):
            t = n_tokens(u) + 1
            if cur and cur_tok + t > budget:
                flush()
                cur_ctx = crumb()
                budget = size - n_tokens(cur_ctx) - 1
            cur.append(u)
            cur_tok += t
            cur_pages.append(b["page"])
    flush()
    return chunks


# ---------------------------------------------------------------- sortie

def _make(doc, d, context, text, pages, section):
    return {"doc": doc, "page_start": pages[0], "page_end": pages[1],
            "section": section, "context": context, "text": text,
            "n_tokens": n_tokens(f"{context}\n{text}")}


def build(strategy: str, size: int) -> list[dict]:
    docs = load_corpus()
    fn = chunk_fixed if strategy == "fixed" else chunk_structural
    chunks = []
    for name, d in docs.items():
        for k, c in enumerate(fn(name, d, size)):
            chunks.append({"chunk_id": f"{name}#{k:04d}", **c})
    return chunks


def save(chunks: list[dict], strategy: str, size: int) -> Path:
    CHUNK_DIR.mkdir(parents=True, exist_ok=True)
    path = CHUNK_DIR / f"{strategy}_{size}.jsonl"
    path.write_text("\n".join(json.dumps(c, ensure_ascii=False) for c in chunks), encoding="utf-8")
    return path


def main() -> None:
    import statistics as st
    print(f"{'config':18} {'chunks':>7} {'tok moy':>8} {'tok méd':>8} {'tok max':>8} {'< 50 tok':>9}")
    for strategy in ("fixed", "structural"):
        for size in (256, 512, 1024):
            ch = build(strategy, size)
            save(ch, strategy, size)
            t = [c["n_tokens"] for c in ch]
            print(f"{strategy + '_' + str(size):18} {len(ch):7} {st.mean(t):8.0f} {st.median(t):8.0f} "
                  f"{max(t):8} {sum(x < 50 for x in t):9}")
    print("nettoyage commun :", load_corpus.stats)


if __name__ == "__main__":
    main()

**`src/bm25.py`** — BM25 codé à la main

In [ ]:
%%writefile src/bm25.py
"""Recherche lexicale BM25, codée à la main (système S1 et brique du système hybride S3).

BM25 note un chunk d pour une requête q ainsi :
    score(q, d) = somme sur les mots t de q de
        idf(t) * tf(t,d) * (k1 + 1) / ( tf(t,d) + k1 * (1 - b + b * |d| / avgdl) )
    idf(t) = ln( (N - df(t) + 0.5) / (df(t) + 0.5) + 1 )
- tf(t,d)  : nombre d'occurrences du mot t dans d (saturé par k1 : la 10e occurrence compte peu) ;
- idf(t)   : un mot rare est plus informatif qu'un mot courant ;
- |d|/avgdl: normalisation par la longueur (b = 0 la désactive, b = 1 la rend totale).
"""
import re
import unicodedata

import numpy as np
from scipy import sparse

STOPWORDS = set("""a au aux avec ce ces dans de des du elle en et eux il ils je la le les leur lui ma mais me meme
mes moi mon ne nos notre nous on ou par pas pour qu que qui sa se ses son sur ta te tes toi ton tu un une vos votre
vous c d j l m n s t y est sont etre ete ont avoir peut doit sans sous si ainsi alors aussi car donc lors""".split())


def tokenize(text: str) -> list[str]:
    """Minuscules, accents supprimés (« machine »/« machiné » se rejoignent), mots vides retirés."""
    text = unicodedata.normalize("NFKD", text.lower())
    text = "".join(c for c in text if not unicodedata.combining(c))
    return [t for t in re.findall(r"[a-z0-9]+", text) if len(t) > 1 and t not in STOPWORDS]


class BM25:
    def __init__(self, texts: list[str], k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        docs = [tokenize(t) for t in texts]
        self.vocab = {}
        rows, cols, vals = [], [], []
        for i, toks in enumerate(docs):
            counts = {}
            for t in toks:
                counts[t] = counts.get(t, 0) + 1
            for t, c in counts.items():
                rows.append(i)
                cols.append(self.vocab.setdefault(t, len(self.vocab)))
                vals.append(c)
        self.tf = sparse.csc_matrix((vals, (rows, cols)), shape=(len(docs), len(self.vocab)), dtype=np.float32)
        self.dl = np.array([len(d) for d in docs], dtype=np.float32)
        self.avgdl = float(self.dl.mean())
        df = np.diff(self.tf.indptr)                       # nb de documents contenant chaque mot
        n = len(docs)
        self.idf = np.log((n - df + 0.5) / (df + 0.5) + 1).astype(np.float32)
        self.norm = k1 * (1 - b + b * self.dl / self.avgdl)  # terme de longueur, un par document

    def scores(self, query: str) -> np.ndarray:
        s = np.zeros(len(self.dl), dtype=np.float32)
        for t in set(tokenize(query)):
            j = self.vocab.get(t)
            if j is None:
                continue
            col = self.tf.getcol(j)
            idx, tf = col.indices, col.data
            s[idx] += self.idf[j] * tf * (self.k1 + 1) / (tf + self.norm[idx])
        return s

    def search(self, query: str, k: int = 10) -> list[tuple[int, float]]:
        s = self.scores(query)
        top = np.argpartition(-s, min(k, len(s) - 1))[:k]
        top = top[np.argsort(-s[top])]
        return [(int(i), float(s[i])) for i in top if s[i] > 0]

**`src/dense.py`** — Embeddings bge-m3 + index FAISS

In [ ]:
%%writefile src/dense.py
"""Recherche dense (système S2) : embeddings bge-m3 + index FAISS, avec cache disque.

Chaque chunk et chaque question devient un vecteur normalisé ; la similarité cosinus
(= produit scalaire sur vecteurs normalisés) mesure la proximité de sens, même sans
mot en commun. Index FAISS « plat » (IndexFlatIP) : recherche exacte, suffisante
pour quelques milliers de chunks.
"""
import gc
from pathlib import Path

import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

ROOT = Path(__file__).resolve().parents[1]
EMB_DIR = ROOT / "data" / "processed" / "embeddings"
MODEL_NAME = "BAAI/bge-m3"
BATCH = 16            # raisonnable pour 6 Go de VRAM avec des chunks <= 1024 tokens en fp16

_model = None


def model() -> SentenceTransformer:
    global _model
    if _model is None:
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        _model = SentenceTransformer(MODEL_NAME, device=dev)
        _model.max_seq_length = 1100
        if dev == "cuda":
            _model.half()
    return _model


def free_gpu() -> None:
    global _model
    _model = None
    gc.collect()
    torch.cuda.empty_cache()


def encode(texts: list[str], batch: int = BATCH) -> np.ndarray:
    # tri par longueur : moins de padding, donc plus rapide et moins de VRAM
    order = np.argsort([-len(t) for t in texts])
    emb = model().encode([texts[i] for i in order], batch_size=batch, normalize_embeddings=True,
                         convert_to_numpy=True, show_progress_bar=False).astype(np.float32)
    out = np.empty_like(emb)
    out[order] = emb
    return out


class DenseIndex:
    def __init__(self, chunks: list[dict], name: str):
        EMB_DIR.mkdir(parents=True, exist_ok=True)
        path = EMB_DIR / f"{MODEL_NAME.split('/')[-1]}_{name}.npy"
        if path.exists():
            self.emb = np.load(path)
        else:
            self.emb = encode([c["context"] + "\n" + c["text"] for c in chunks])
            np.save(path, self.emb)
        self.index = faiss.IndexFlatIP(self.emb.shape[1])
        self.index.add(self.emb)

    def search_emb(self, q_emb: np.ndarray, k: int = 10):
        scores, idx = self.index.search(q_emb, k)
        return idx, scores

    def search(self, query: str, k: int = 10) -> list[tuple[int, float]]:
        idx, sc = self.search_emb(encode([query]), k)
        return list(zip(idx[0].tolist(), sc[0].tolist()))

**`src/hybrid.py`** — Fusion RRF et reranker cross-encoder

In [ ]:
%%writefile src/hybrid.py
"""Recherche hybride + reranking (système S3).

1) Fusion RRF (Reciprocal Rank Fusion) : BM25 et dense produisent chacun un classement ;
   un chunk reçoit  score = somme sur les classements de 1 / (K + rang)  (K = 60, valeur usuelle).
   On ne compare jamais les scores bruts (échelles différentes), seulement les rangs : un chunk
   bien classé par les DEUX méthodes remonte.
2) Reranker cross-encoder : contrairement aux embeddings (question et chunk encodés séparément),
   le cross-encoder lit le couple (question, chunk) ensemble et produit un score de pertinence.
   Plus précis mais plus lent : on ne l'applique qu'aux N meilleurs candidats de la fusion.
"""
import gc

import torch
from sentence_transformers import CrossEncoder

RRF_K = 60
RERANKER = "BAAI/bge-reranker-v2-m3"
_ce = None


def rrf(rankings: list[list[int]], k: int = RRF_K, top: int = 30) -> list[int]:
    scores = {}
    for ranking in rankings:
        for rank, i in enumerate(ranking, 1):
            scores[i] = scores.get(i, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)[:top]


def reranker() -> CrossEncoder:
    global _ce
    if _ce is None:
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        _ce = CrossEncoder(RERANKER, device=dev, max_length=1024,
                           model_kwargs={"torch_dtype": torch.float16} if dev == "cuda" else {})
    return _ce


def free_gpu() -> None:
    global _ce
    _ce = None
    gc.collect()
    torch.cuda.empty_cache()


def rerank(query: str, chunks: list[dict], candidates: list[int], k: int = 10, batch: int = 8) -> list[int]:
    pairs = [(query, chunks[i]["context"] + "\n" + chunks[i]["text"]) for i in candidates]
    sc = reranker().predict(pairs, batch_size=batch, show_progress_bar=False)
    order = sorted(range(len(candidates)), key=lambda j: -sc[j])
    return [candidates[j] for j in order[:k]]

**`src/retrieval_eval.py`** — Recall@k et MRR indépendants du découpage

In [ ]:
%%writefile src/retrieval_eval.py
"""Évaluation du retrieval : Recall@k et MRR, indépendamment de la façon dont le corpus est découpé.

Un chunk est « pertinent » pour une preuve si, dans le même document, il contient au moins
MIN_COVER de la preuve (part des triplets de mots de la preuve retrouvés dans le chunk).
Cette définition fonctionne pour toutes les tailles de chunks, y compris quand une preuve est
coupée entre deux chunks fixes (chacun la couvre alors en partie).

- Recall@k : part des preuves d'une question couvertes par les k premiers chunks
  (pour une question multi-passages à 2 preuves : 0, 0.5 ou 1), moyennée sur les questions.
- MRR : 1 / rang du premier chunk pertinent (0 s'il n'y en a aucun dans le top-K).
Les questions sans réponse n'ont pas de preuve : elles sont exclues du retrieval.
"""
import json
import re
from pathlib import Path

import numpy as np

from src.testset import norm

ROOT = Path(__file__).resolve().parents[1]
CHUNK_DIR = ROOT / "data" / "processed" / "chunks"
MIN_COVER = 0.6
KS = (1, 3, 5, 10)


def load_chunks(name: str) -> list[dict]:
    return [json.loads(l) for l in (CHUNK_DIR / f"{name}.jsonl").read_text(encoding="utf-8").splitlines()]


def _trigrams(text: str) -> set:
    w = re.findall(r"\w+", norm(text))
    return {tuple(w[i:i + 3]) for i in range(len(w) - 2)} or {tuple(w)}


class Gold:
    """Pré-calcule, pour une configuration de chunks, quels chunks couvrent chaque preuve."""

    def __init__(self, chunks: list[dict], questions: list[dict]):
        by_doc = {}
        for i, c in enumerate(chunks):
            by_doc.setdefault(c["doc"], []).append((i, _trigrams(c["text"])))
        self.rel = {}                                   # id question -> liste (une par preuve) d'ensembles d'indices
        for q in questions:
            per_ev = []
            for e in q["preuves"]:
                tg = _trigrams(e["quote"])
                per_ev.append({i for i, ctg in by_doc.get(e["doc"], [])
                               if len(tg & ctg) / len(tg) >= MIN_COVER})
            self.rel[q["id"]] = per_ev

    def unreachable(self) -> list[str]:
        return [q for q, evs in self.rel.items() if any(not s for s in evs)]


def evaluate(rank_fn, chunks: list[dict], questions: list[dict], ks=KS, kmax=10) -> dict:
    """rank_fn(question: str, k: int) -> liste d'indices de chunks (meilleur d'abord)."""
    qs = [q for q in questions if q["type"] != "sans_reponse"]
    gold = Gold(chunks, qs)
    per_q = []
    for q in qs:
        top = rank_fn(q["question"], kmax)
        evs = gold.rel[q["id"]]
        row = {"id": q["id"], "type": q["type"]}
        for k in ks:
            topk = set(top[:k])
            row[f"recall@{k}"] = np.mean([bool(s & topk) for s in evs])
        rr = 0.0
        for rank, i in enumerate(top, 1):
            if any(i in s for s in evs):
                rr = 1 / rank
                break
        row["mrr"] = rr
        per_q.append(row)
    return {"per_question": per_q, "n": len(per_q), "unreachable": gold.unreachable()}


def summarize(res: dict) -> dict:
    rows = res["per_question"]
    keys = [k for k in rows[0] if k not in ("id", "type")]
    return {k: float(np.mean([r[k] for r in rows])) for k in keys}

**`src/llm.py`** — Client LLM avec cache disque

In [ ]:
%%writefile src/llm.py
"""Client LLM (API Groq) avec cache disque, reprise sur limites de débit et mesure des coûts.

Le cache est la clé de la reproductibilité : une même requête (modèle, messages,
paramètres) n'est jamais payée deux fois, et relancer le notebook redonne les
mêmes réponses. Chaque appel renvoie aussi le nombre de tokens et la latence,
utilisés plus tard pour l'analyse des coûts.
"""
import hashlib
import json
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from groq import Groq, RateLimitError, APIConnectionError, APIStatusError

ROOT = Path(__file__).resolve().parents[1]
CACHE_DIR = ROOT / "data" / "processed" / "llm_cache"
MAX_WAIT = 90          # au-delà (limite journalière atteinte), on s'arrête au lieu de dormir des heures

load_dotenv(ROOT / ".env")
_client = None


def client() -> Groq:
    global _client
    if _client is None:
        key = os.getenv("GROQ_API_KEY")
        if not key:
            raise RuntimeError("GROQ_API_KEY absente : la renseigner dans .env")
        _client = Groq(api_key=key, max_retries=0)      # les reprises sont gérées ici
    return _client


def _key(model: str, messages: list[dict], params: dict) -> str:
    blob = json.dumps({"m": model, "msg": messages, "p": params}, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()


def chat(messages: list[dict], model: str, temperature: float = 0.0, max_tokens: int = 1024,
         json_mode: bool = False, reasoning_effort: str | None = None, seed: int = 0) -> dict:
    """Un appel de chat. Renvoie {text, prompt_tokens, completion_tokens, latency_s, cached, model}."""
    params = {"temperature": temperature, "max_tokens": max_tokens, "json": json_mode,
              "reasoning_effort": reasoning_effort, "seed": seed}
    path = CACHE_DIR / f"{_key(model, messages, params)}.json"
    if path.exists():
        return {**json.loads(path.read_text(encoding="utf-8")), "cached": True}

    kwargs = dict(model=model, messages=messages, temperature=temperature,
                  max_completion_tokens=max_tokens, seed=seed)
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    if reasoning_effort:
        kwargs["reasoning_effort"] = reasoning_effort

    for attempt in range(8):
        t0 = time.time()
        try:
            r = client().chat.completions.create(**kwargs)
            break
        except RateLimitError as e:
            wait = float(e.response.headers.get("retry-after", 2 ** attempt * 2))
            if wait > MAX_WAIT:
                raise RuntimeError(f"Limite de débit atteinte, attente demandée {wait:.0f}s : {e}") from e
            time.sleep(wait + 1)
        except (APIConnectionError, APIStatusError) as e:
            if isinstance(e, APIStatusError) and e.status_code < 500:
                raise
            time.sleep(2 ** attempt)
    else:
        raise RuntimeError("Échec après plusieurs tentatives")

    out = {"text": (r.choices[0].message.content or "").strip(),
           "prompt_tokens": r.usage.prompt_tokens, "completion_tokens": r.usage.completion_tokens,
           "latency_s": round(time.time() - t0, 3), "model": model}
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(out, ensure_ascii=False), encoding="utf-8")
    return {**out, "cached": False}

**`src/testset.py`** — Génération et vérification des questions

In [ ]:
%%writefile src/testset.py
"""Génération du jeu de test (questions de référence) avec un LLM, puis vérification automatique.

Chaque question vient d'un passage réel du corpus. Le LLM doit recopier MOT POUR MOT
l'extrait qui prouve la réponse (« preuve ») ; on vérifie que cet extrait existe bien
dans le document source et on en retrouve la page. Les questions dont la preuve n'est
pas retrouvée sont rejetées automatiquement. Le résultat est ensuite relu à la main
(fichier CSV) : le LLM propose, l'humain valide.

Types de questions : factuelle, numérique/tabulaire, multi-passages (deux sources à
combiner) et sans réponse dans le corpus (pour tester l'abstention).

Usage : python -m src.testset            (génération complète, reprend grâce au cache)
        python -m src.testset --quick    (2 questions par type, pour juger la qualité)
"""
import argparse
import json
import math
import random
import re
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

from src import llm

ROOT = Path(__file__).resolve().parents[1]
CHUNKS = ROOT / "data" / "processed" / "chunks" / "structural_512.jsonl"
TEXT_DIR = ROOT / "data" / "processed" / "text"
OUT_DIR = ROOT / "data" / "testset"

SEED = 42
GENERATORS = ["openai/gpt-oss-120b", "qwen/qwen3.8-27b"]     # 2e = repli si le quota journalier du 1er est atteint
# brouillons visés par type (on en génère plus que les ~150 finaux : la relecture en éliminera)
TARGETS = {"factuelle": 90, "numerique": 55, "multi": 45, "sans_reponse": 35}
STYLES = ["Quel / Quelle", "Comment", "Dans quel cas", "Que doit faire / Que doivent faire",
          "Qui est responsable de", "À quelle condition", "Quelle différence entre", "Que signifie", "Combien"]
BANNED = re.compile(r"\b(l'extrait|le passage|le texte|ce texte|ce document|le document|ci-dessus|cet extrait)\b", re.I)

SYSTEM = ("Tu es un expert en sécurité des machines industrielles. Tu construis un jeu de test pour évaluer "
          "un système de questions-réponses sur des documents réglementaires et techniques (INRS, "
          "règlement machines UE 2023/1230, directive 2006/42/CE). Tu réponds uniquement par un objet JSON.")

RULES = """Règles pour la question :
- Elle doit être AUTONOME : compréhensible sans voir l'extrait. N'écris jamais « l'extrait », « le texte », « le passage » ni « le document ».
- Elle doit contenir ce qui l'identifie sans ambiguïté (type de machine, équipement, article, norme, étape...) : une question comme « Quelle est la fréquence ? » est interdite.
- Elle doit porter sur un point précis, pas sur le sujet général. Style suggéré : {style}.
- Écris en français naturel, comme un préventeur, un concepteur ou un chef d'établissement.
- La réponse de référence ne contient QUE des faits écrits explicitement dans l'extrait : aucune explication, justification ni généralité ajoutée de ton propre chef.
- Si l'extrait ne permet pas une question utile et précise (case de formulaire, liste de références, en-tête, contenu trop vague), réponds {{"question": null}}."""


# ---------------------------------------------------------------- outils texte

def norm(t: str) -> str:
    """Forme de comparaison : apostrophes/espaces/casse neutralisés."""
    t = unicodedata.normalize("NFKC", t).replace("’", "'").replace("‘", "'").replace("«", '"').replace("»", '"')
    return re.sub(r"\s+", " ", t).strip().lower()


def find_page(doc: str, quote: str, default: int) -> int:
    """Page du document où se trouve la preuve (sinon page de début du chunk)."""
    q = norm(quote)
    d = json.loads((TEXT_DIR / f"{doc}.json").read_text(encoding="utf-8"))
    for p in d["pages"]:
        if q in norm(p["text"]):
            return p["page"]
    return default


def parse_json(text: str) -> dict | None:
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    m = re.search(r"\{.*\}", text, flags=re.S)
    try:
        return json.loads(m.group()) if m else None
    except json.JSONDecodeError:
        return None


# ---------------------------------------------------------------- sélection des passages sources

NUM_RE = re.compile(r"\d+(?:[.,]\d+)?\s?(?:mm|cm|m|s|ms|kg|n|%|hz|db|v|a|mois|ans?|jours?|heures?|h|bar|pl[a-e]|sil)\b", re.I)


def load_pool() -> list[dict]:
    chunks = [json.loads(l) for l in CHUNKS.read_text(encoding="utf-8").splitlines()]
    pool = []
    for c in chunks:
        sec = c["section"].lower()
        letters = sum(ch.isalpha() for ch in c["text"]) / max(1, len(c["text"]))
        if (c["n_tokens"] < 150 or letters < 0.6 or "bibliograph" in sec or "sommaire" in sec
                or c["text"].count("☐") > 2                      # formulaires à cocher
                or re.search(r"isbn|dépôt légal|reproduction", c["text"], re.I)):
            continue
        rows = sum(" : " in l and " | " in l for l in c["text"].split("\n"))
        c["numeric"] = rows >= 3 or len(NUM_RE.findall(c["text"])) >= 4
        pool.append(c)
    return pool


def sample(pool: list[dict], n: int, rng: random.Random, used: set) -> list[dict]:
    """n chunks répartis sur les documents (poids ~ racine de la taille du document), sans répétition."""
    by_doc = {}
    for c in pool:
        if c["chunk_id"] not in used:
            by_doc.setdefault(c["doc"], []).append(c)
    for v in by_doc.values():
        rng.shuffle(v)
    out = []
    while len(out) < n and by_doc:
        docs = list(by_doc)
        d = rng.choices(docs, weights=[math.sqrt(len(by_doc[k])) for k in docs])[0]
        c = by_doc[d].pop()
        if not by_doc[d]:
            del by_doc[d]
        out.append(c)
        used.add(c["chunk_id"])
    return out


def neighbours(pool: list[dict]):
    """Matrice de similarité TF-IDF entre chunks (pour trouver un second passage lié au premier)."""
    vec = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), min_df=2, max_df=0.5)
    X = vec.fit_transform([c["text"] for c in pool])
    return (X @ X.T).toarray()


# ---------------------------------------------------------------- prompts

def passage(c: dict) -> str:
    return f"[{c['context']}, page {c['page_start']}]\n{c['text']}"


def prompt_single(kind: str, c: dict, style: str) -> str:
    focus = ("La question doit demander une VALEUR PRÉCISE (nombre, seuil, durée, fréquence, distance, niveau, "
             "classe...) ou une information à lire dans un tableau ; la réponse contient cette valeur avec son unité."
             if kind == "numerique" else
             "La question doit avoir une réponse trouvable explicitement dans l'extrait.")
    return (f"Voici un extrait d'un document :\n\"\"\"\n{passage(c)}\n\"\"\"\n\n"
            f"Rédige UNE question dont la réponse se trouve dans cet extrait. {focus}\n\n"
            f"{RULES.format(style=style)}\n"
            "- « reponse » : 1 à 3 phrases, fondée uniquement sur l'extrait.\n"
            "- « preuve » : copie MOT POUR MOT (copier-coller exact, rien modifié) la ou les 1-2 phrases de l'extrait "
            "qui justifient la réponse.\n\n"
            'Réponds par : {"question": "...", "reponse": "...", "preuve": "..."}')


def prompt_multi(a: dict, b: dict, style: str) -> str:
    return (f"Voici deux extraits.\n\nEXTRAIT A :\n\"\"\"\n{passage(a)}\n\"\"\"\n\nEXTRAIT B :\n\"\"\"\n{passage(b)}\n\"\"\"\n\n"
            "Rédige UNE question qui nécessite de COMBINER une information de A et une information de B pour "
            "répondre (ni A seul ni B seul ne suffisent). Le lien entre A et B doit être naturel et porter sur le "
            "même sujet précis (même équipement, même obligation, même notion). Si ce lien n'existe pas, "
            'réponds {"question": null} : ne force pas un rapprochement artificiel.\n\n'
            f"{RULES.format(style=style)}\n"
            "- « reponse » : 2 à 4 phrases, fondée uniquement sur A et B.\n"
            "- « preuve_a » et « preuve_b » : copie MOT POUR MOT la phrase de A, puis celle de B, qui servent à répondre.\n\n"
            'Réponds par : {"question": "...", "reponse": "...", "preuve_a": "...", "preuve_b": "..."}')


def prompt_noanswer(c: dict, angle: str, style: str) -> str:
    return (f"Voici un extrait d'un document sur la sécurité des machines :\n\"\"\"\n{passage(c)}\n\"\"\"\n\n"
            "Rédige UNE question réaliste, du même domaine et du même sujet, que ce document NE PERMET PAS de résoudre : "
            f"{angle}. La question doit paraître légitime et précise (pas absurde, pas hors sujet), sans que la "
            "réponse figure dans l'extrait ni puisse se déduire de lui.\n\n"
            f"{RULES.format(style=style)}\n\n"
            'Réponds par : {"question": "...", "pourquoi_sans_reponse": "..."}')


ANGLES = ["elle demande une valeur chiffrée, un seuil ou une durée précis qui n'est pas mentionné",
          "elle porte sur un autre type de machine ou d'équipement que ceux décrits",
          "elle porte sur une règle propre à un secteur d'activité (agriculture, bâtiment, mines...) que l'extrait ne traite pas",
          "elle demande un exemple concret, un cas d'application ou une marque que l'extrait ne donne pas",
          "elle porte sur une exigence d'une norme ou d'un texte que l'extrait ne cite pas"]


# ---------------------------------------------------------------- génération

def call(prompt: str, state: dict) -> dict | None:
    """Appelle le générateur courant ; bascule sur le suivant si le quota journalier est atteint."""
    while state["g"] < len(GENERATORS):
        model = GENERATORS[state["g"]]
        try:
            r = llm.chat([{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}], model=model,
                         temperature=0.7, max_tokens=900, json_mode=True,
                         reasoning_effort="low" if "gpt-oss" in model else None, seed=state["seed"])
        except RuntimeError as e:
            print(f"  ! {model} indisponible ({str(e)[:90]}) -> générateur suivant")
            state["g"] += 1
            continue
        r["parsed"] = parse_json(r["text"])
        return r
    return None


def build_item(kind, r, chunks, quotes, extra=None):
    p = r["parsed"]
    if p and p.get("question") is None:
        return None, "ignoree_par_le_llm"
    if not p or not p.get("question") or not p.get("reponse" if kind != "sans_reponse" else "question"):
        return None, "json_invalide"
    q = p["question"].strip()
    if BANNED.search(norm(q)):
        return None, "question_non_autonome"
    if not (25 <= len(q) <= 300) or not q.endswith("?"):
        return None, "question_forme"
    ev = []
    for quote, c in zip(quotes, chunks):
        if kind == "sans_reponse":
            break
        quote = (quote or "").strip()
        if len(quote) < 20 or norm(quote) not in norm(c["text"]):
            return None, "preuve_introuvable"
        ev.append({"doc": c["doc"], "page": find_page(c["doc"], quote, c["page_start"]), "quote": quote,
                   "chunk_id_structural_512": c["chunk_id"]})
    item = {"type": kind, "question": q, "reponse": (p.get("reponse") or "").strip() or None,
            "preuves": ev, "generateur": r["model"], "prompt_tokens": r["prompt_tokens"],
            "completion_tokens": r["completion_tokens"],
            "source_chunks": [c["chunk_id"] for c in chunks]}
    if kind == "sans_reponse":
        item["pourquoi_sans_reponse"] = p.get("pourquoi_sans_reponse")
        item["reponse"] = None
    return item, "ok"


def generate(targets: dict) -> list[dict]:
    rng = random.Random(SEED)
    pool = load_pool()
    sim = neighbours(pool)
    idx = {c["chunk_id"]: i for i, c in enumerate(pool)}
    used, items, rejects = set(), [], Counter()
    state = {"g": 0, "seed": SEED}
    plain = [c for c in pool if not c["numeric"]]
    numeric = [c for c in pool if c["numeric"]]
    print(f"pool : {len(pool)} chunks ({len(numeric)} numériques/tabulaires, {len(plain)} textuels)")

    plan = []
    for c in sample(plain, targets["factuelle"], rng, used):
        plan.append(("factuelle", [c]))
    for c in sample(numeric, targets["numerique"], rng, used):
        plan.append(("numerique", [c]))
    for a in sample(pool, targets["multi"], rng, used):
        row = sim[idx[a["chunk_id"]]]
        cand = [j for j in np.argsort(-row)[:15]
                if 0.12 <= row[j] <= 0.65 and pool[j]["chunk_id"] != a["chunk_id"]
                and not (pool[j]["doc"] == a["doc"] and abs(j - idx[a["chunk_id"]]) <= 1)]
        if cand:
            plan.append(("multi", [a, pool[rng.choice(cand)]]))
    for c in sample(plain, targets["sans_reponse"], rng, used):
        plan.append(("sans_reponse", [c]))
    rng.shuffle(plan)

    for n, (kind, chunks) in enumerate(plan, 1):
        style = rng.choice(STYLES)
        if kind == "multi":
            prompt = prompt_multi(chunks[0], chunks[1], style)
        elif kind == "sans_reponse":
            prompt = prompt_noanswer(chunks[0], rng.choice(ANGLES), style)
        else:
            prompt = prompt_single(kind, chunks[0], style)
        r = call(prompt, state)
        if r is None:
            print("Plus aucun générateur disponible : on s'arrête, relancer plus tard (le cache est conservé).")
            break
        p = r["parsed"] or {}
        quotes = [p.get("preuve_a"), p.get("preuve_b")] if kind == "multi" else [p.get("preuve")]
        item, why = build_item(kind, r, chunks, quotes)
        rejects[why] += 1
        if item:
            items.append(item)
        if n % 10 == 0 or n == len(plan):
            print(f"  {n}/{len(plan)} appels | gardées {len(items)} | {dict(rejects)}")
    return items


def dedupe(items: list[dict], thr: float = 0.75) -> list[dict]:
    """Retire les questions quasi identiques (similarité TF-IDF > thr)."""
    if len(items) < 2:
        return items
    X = TfidfVectorizer(sublinear_tf=True).fit_transform([i["question"] for i in items])
    S = (X @ X.T).toarray()
    keep = []
    for i in range(len(items)):
        if all(S[i, j] <= thr for j in keep):
            keep.append(i)
    return [items[i] for i in keep]


REVIEW_COLS = ["id", "type", "question", "reponse_reference", "source", "preuve", "passages_proches",
               "verdict", "question_corrigee", "reponse_corrigee", "commentaire"]


def export_review() -> Path:
    """CSV de relecture. Pour les questions « sans réponse », on ajoute les 3 passages du corpus les plus
    proches (BM25) : c'est là qu'il faut vérifier que la réponse n'existe pas."""
    import csv
    from src.bm25 import BM25
    chunks = [json.loads(l) for l in CHUNKS.read_text(encoding="utf-8").splitlines()]
    bm = BM25([c["context"] + "\n" + c["text"] for c in chunks])
    items = [json.loads(l) for l in (OUT_DIR / "brouillon.jsonl").read_text(encoding="utf-8").splitlines()]
    order = {"factuelle": 0, "numerique": 1, "multi": 2, "sans_reponse": 3}
    items.sort(key=lambda i: (order[i["type"]], i["id"]))
    path = OUT_DIR / "questions_a_verifier.csv"
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=REVIEW_COLS, delimiter=";")
        w.writeheader()
        for it in items:
            near = ""
            if it["type"] == "sans_reponse":
                near = "\n".join(f"[{chunks[i]['doc']} p.{chunks[i]['page_start']}] {chunks[i]['text'][:260]}"
                                 for i, _ in bm.search(it["question"], 3))
            w.writerow({"id": it["id"], "type": it["type"], "question": it["question"],
                        "reponse_reference": it["reponse"] or "(aucune : la question n'a pas de réponse dans le corpus)",
                        "source": " | ".join(f"{e['doc']} p.{e['page']}" for e in it["preuves"]) or "-",
                        "preuve": "\n".join(e["quote"] for e in it["preuves"]), "passages_proches": near,
                        "verdict": "", "question_corrigee": "", "reponse_corrigee": "", "commentaire": ""})
    return path


def finalize() -> Path:
    """Reprend le CSV relu : OK = gardée, CORRIGER = gardée avec la correction, KO (ou vide) = écartée."""
    import csv
    drafts = {i["id"]: i for i in map(json.loads, (OUT_DIR / "brouillon.jsonl").read_text(encoding="utf-8").splitlines())}
    final, skipped = [], Counter()
    with open(OUT_DIR / "questions_a_verifier.csv", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f, delimiter=";"):
            v = row["verdict"].strip().upper()
            it = drafts[row["id"]]
            if v not in ("OK", "CORRIGER"):
                skipped[v or "non relue"] += 1
                continue
            it = {k: it[k] for k in ("id", "type", "question", "reponse", "preuves", "generateur", "source_chunks")}
            tidy = str.maketrans({" ": " ", " ": " ", "‑": "-", "’": "'"})   # espaces/traits insécables des LLM
            it["question"] = it["question"].translate(tidy)
            it["reponse"] = it["reponse"].translate(tidy) if it["reponse"] else None
            if v == "CORRIGER":
                it["question"] = row["question_corrigee"].strip() or it["question"]
                if it["type"] != "sans_reponse":
                    it["reponse"] = row["reponse_corrigee"].strip() or it["reponse"]
                it["corrigee_a_la_main"] = True
            it["note"] = row["commentaire"].strip()
            final.append(it)
    path = OUT_DIR / "questions.jsonl"
    path.write_text("\n".join(json.dumps(i, ensure_ascii=False) for i in final), encoding="utf-8")
    print(f"{len(final)} questions validées -> {path.name} | par type : {dict(Counter(i['type'] for i in final))}"
          f" | écartées : {dict(skipped)}")
    return path


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", nargs="?", default="generate", choices=["generate", "export", "finalize"])
    ap.add_argument("--quick", action="store_true", help="2 questions par type")
    args = ap.parse_args()
    if args.cmd == "export":
        print("CSV de relecture :", export_review())
        return
    if args.cmd == "finalize":
        finalize()
        return
    targets = {k: 2 for k in TARGETS} if args.quick else TARGETS
    items = dedupe(generate(targets))
    for k, it in enumerate(items, 1):
        it["id"] = f"Q{k:03d}"
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    path = OUT_DIR / ("brouillon_quick.jsonl" if args.quick else "brouillon.jsonl")
    path.write_text("\n".join(json.dumps(i, ensure_ascii=False) for i in items), encoding="utf-8")
    print(f"\n{len(items)} questions -> {path.name} | par type : {dict(Counter(i['type'] for i in items))}")


if __name__ == "__main__":
    main()

**`src/rag.py`** — Prompts, génération, F1, abstention, juge LLM

In [ ]:
%%writefile src/rag.py
"""Génération des réponses (S0 à S3) et évaluation : F1, abstention, juge LLM.

Modèles (Groq, open-weights) :
  - générateur : openai/gpt-oss-20b  (petit, rapide, quota propre)
  - juge       : qwen/qwen3.8-27b    (autre famille que le générateur, pour limiter le biais d'auto-préférence)
"""
import json
import re
import unicodedata
from collections import Counter

from src import llm
from src.testset import parse_json

GENERATOR = "openai/gpt-oss-20b"
JUDGE = "qwen/qwen3.8-27b"
ABSTAIN = "Information absente des documents."

PROMPT_RAG_SIMPLE = ("Tu es un assistant expert en sécurité des machines. Réponds à la question en français, "
                     "de façon concise, en t'aidant des extraits fournis.")
PROMPT_RAG_CITE = ("Tu es un assistant expert en sécurité des machines. Réponds en français, de façon concise (1 à 3 phrases), "
                   "UNIQUEMENT à partir des extraits fournis. Cite les extraits utilisés entre crochets, par exemple [1] ou [2]. "
                   f"Si les extraits ne contiennent pas la réponse, réponds exactement : « {ABSTAIN} » sans rien ajouter.")
PROMPT_CLOSED = ("Tu es un assistant expert en sécurité des machines industrielles. Réponds de mémoire, en français, de façon "
                 "concise (1 à 3 phrases). Si tu n'es pas certain de la réponse, réponds exactement : « Je ne sais pas. »")


def format_context(chunks: list[dict]) -> str:
    return "\n\n".join(f"[{n}] ({c['doc']}, p.{c['page_start']})\n{c['text']}" for n, c in enumerate(chunks, 1))


def answer(question: str, chunks: list[dict] | None, prompt: str = PROMPT_RAG_CITE, model: str = GENERATOR) -> dict:
    if chunks is None:                                   # S0 : sans documents
        msgs = [{"role": "system", "content": PROMPT_CLOSED}, {"role": "user", "content": question}]
    else:
        msgs = [{"role": "system", "content": prompt},
                {"role": "user", "content": f"Extraits :\n{format_context(chunks)}\n\nQuestion : {question}"}]
    r = llm.chat(msgs, model=model, temperature=0.0, max_tokens=600, reasoning_effort="low")
    return {k: r[k] for k in ("text", "prompt_tokens", "completion_tokens", "latency_s", "cached")}


# ---------------------------------------------------------------- métriques automatiques

def _tok(t: str) -> list[str]:
    t = unicodedata.normalize("NFKD", t.lower().replace("’", "'"))
    t = "".join(c for c in t if not unicodedata.combining(c))
    return re.findall(r"[a-z0-9]+", re.sub(r"\[\d+\]", " ", t))


def f1(pred: str, ref: str) -> float:
    p, r = _tok(pred), _tok(ref)
    common = sum((Counter(p) & Counter(r)).values())
    if not p or not r or common == 0:
        return 0.0
    pr, rc = common / len(p), common / len(r)
    return 2 * pr * rc / (pr + rc)


def is_abstention(text: str) -> bool:
    t = unicodedata.normalize("NFKD", text.lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    return bool(re.search(r"information absente|je ne sais pas|ne (?:permettent|contiennent|figure|mentionnent)", t))


def cited_ids(text: str) -> list[int]:
    return sorted({int(x) for x in re.findall(r"\[(\d+)\]", text)})


def citation_ok(text: str, n_chunks: int) -> bool | None:
    """Exactitude formelle des citations : toutes les références existent. None si aucune citation."""
    ids = cited_ids(text)
    return None if not ids else all(1 <= i <= n_chunks for i in ids)


# ---------------------------------------------------------------- juge LLM

JUDGE_SYSTEM = "Tu es un évaluateur rigoureux et impartial. Tu réponds uniquement par un objet JSON."


def judge(question: str, reference: str | None, pred: str, chunks: list[dict] | None) -> dict:
    """Notes du juge. correct : 1 / 0.5 / 0 par rapport à la référence ;
    fidele : 1 / 0 (toutes les affirmations sont soutenues par les extraits) ; None si pas d'extraits."""
    ctx = f"\nEXTRAITS FOURNIS AU SYSTÈME :\n{format_context(chunks)}\n" if chunks else ""
    ask_fid = ('- "fidele" : 1 si toutes les affirmations de la réponse sont soutenues par les extraits fournis, 0 si elle '
               "contient une information absente des extraits (hallucination). Une abstention est fidèle (1).\n"
               if chunks else "")
    p = (f"QUESTION : {question}\nRÉPONSE DE RÉFÉRENCE : {reference}\n{ctx}\nRÉPONSE À ÉVALUER : {pred}\n\n"
         'Évalue : \n- "correct" : 1 si la réponse est équivalente à la référence (mêmes faits clés, valeurs exactes), '
         "0.5 si partiellement correcte ou incomplète, 0 si fausse ou si elle s'abstient.\n"
         f"{ask_fid}"
         'Réponds par : {"correct": ..., "fidele": ..., "raison": "une phrase"}')
    r = llm.chat([{"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": p}],
                 model=JUDGE, temperature=0.0, max_tokens=500, json_mode=True)
    j = parse_json(r["text"]) or {}
    return {"correct": j.get("correct"), "fidele": j.get("fidele") if chunks else None,
            "raison": j.get("raison", ""), "judge_tokens": r["prompt_tokens"] + r["completion_tokens"]}

**`src/analyze.py`** — Bootstrap, taxonomie d'erreurs, figures, kappa

In [ ]:
%%writefile src/analyze.py
"""Analyse des résultats : métriques par système, bootstrap, erreurs, figures, annotation du juge (kappa).

Usage : python -m src.analyze report     [--gen results/generations_provisoire.csv]
        python -m src.analyze annotate   (crée results/annotations_a_remplir.csv, 30 réponses à noter à la main)
        python -m src.analyze kappa      (accord humain / juge, une fois le fichier rempli)
"""
import argparse
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

ROOT = Path(__file__).resolve().parents[1]
RES, FIG = ROOT / "results", ROOT / "figures"
SYSTEMS = ["S0", "S1_bm25", "S2_dense", "S3_hybride_rerank"]
LABEL = {"S0": "S0 sans retrieval", "S1_bm25": "S1 BM25", "S2_dense": "S2 dense", "S3_hybride_rerank": "S3 hybride+rerank"}
SEED, N_BOOT = 42, 5000
# Tarifs indicatifs en $ par million de tokens (à vérifier sur la page de tarification Groq avant de citer).
PRICE = {"in": 0.075, "out": 0.30}


def bootstrap_ci(x, n=N_BOOT, seed=SEED):
    x = np.asarray(x, dtype=float)
    rng = np.random.default_rng(seed)
    means = rng.choice(x, (n, len(x)), replace=True).mean(axis=1)
    return float(x.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def paired_diff(a, b, n=N_BOOT, seed=SEED):
    """IC bootstrap de la différence moyenne a - b sur les MÊMES questions (test apparié)."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    return bootstrap_ci(d, n, seed)


def fmt(m, lo, hi):
    return f"{m:.2f} [{lo:.2f} ; {hi:.2f}]"


def report(gen_path: Path, tag: str = "") -> None:
    global SYSTEMS
    sfx = f"_{tag}" if tag else ""
    g = pd.read_csv(gen_path)
    g["correct_juge"] = pd.to_numeric(g["correct_juge"], errors="coerce")
    counts = g.groupby("systeme").size()
    SYSTEMS = [s for s in SYSTEMS if s in counts.index]
    for s in SYSTEMS:
        print(f"{LABEL[s]} : {counts[s]} questions" + ("  <-- INCOMPLET" if counts[s] < counts.max() else ""))
    excluded = [s for s in SYSTEMS if counts[s] < 0.5 * counts.max()]
    if excluded:
        print(f"Exclus de l'analyse (couverture < 50 %) : {[LABEL[s] for s in excluded]}")
        SYSTEMS = [s for s in SYSTEMS if s not in excluded]
    # comparaison équitable : uniquement les questions traitées par TOUS les systèmes présents
    common = set.intersection(*[set(g[g.systeme == s].id) for s in SYSTEMS])
    g = g[g.id.isin(common)]
    print(f"-> analyse sur les {len(common)} questions communes à tous les systèmes\n")
    ans, noans = g[g.type != "sans_reponse"], g[g.type == "sans_reponse"]
    ret = pd.read_csv(RES / "retrieval_par_question_structural_512.csv")
    rows = []
    for s in SYSTEMS:
        a, n = ans[ans.systeme == s], noans[noans.systeme == s]
        row = {"systeme": LABEL[s], "n_répondables": len(a), "n_sans_réponse": len(n)}
        row["correct (juge)"] = fmt(*bootstrap_ci(a.correct_juge.fillna(0)))
        row["F1"] = fmt(*bootstrap_ci(a.f1.fillna(0)))
        if s != "S0":
            row["fidélité (juge)"] = fmt(*bootstrap_ci(g[g.systeme == s].fidele_juge.dropna()))
        row["abstention correcte"] = fmt(*bootstrap_ci(n.abstention.astype(float))) if len(n) else "-"
        row["refus injustifié"] = f"{a.abstention.mean():.2f}"
        t = g[g.systeme == s]
        if s != "S0":                                  # citations : présence et validité formelle (numéros existants)
            rep = t[~t.abstention.astype(bool)]
            row["réponses citées"] = f"{rep.citation_ok.notna().mean():.2f}"
            row["citations valides"] = f"{rep.citation_ok.dropna().astype(bool).mean():.2f}"
        row["tokens/requête"] = int((t.prompt_tokens + t.completion_tokens).mean())
        row["latence (s)"] = round(t.latency_s.mean(), 2)
        row["coût / 1000 req ($)"] = round((t.prompt_tokens.mean() * PRICE["in"] + t.completion_tokens.mean() * PRICE["out"]) / 1000, 3)
        rows.append(row)
    table = pd.DataFrame(rows)
    table.to_csv(RES / f"metriques_par_systeme{sfx}.csv", index=False)
    print(table.to_string(index=False))

    print("\nDifférences appariées (IC95 % bootstrap, correct juge, questions répondables) :")
    piv = ans.pivot(index="id", columns="systeme", values="correct_juge").fillna(0)
    diffs = []
    for a_, b_ in [("S3_hybride_rerank", "S1_bm25"), ("S3_hybride_rerank", "S2_dense"), ("S1_bm25", "S0"), ("S3_hybride_rerank", "S0")]:
        if a_ in piv and b_ in piv:
            m, lo, hi = paired_diff(piv[a_], piv[b_])
            sig = "significatif" if lo > 0 or hi < 0 else "non significatif"
            print(f"  {LABEL[a_]} - {LABEL[b_]} : {fmt(m, lo, hi)}  -> {sig}")
            diffs.append({"a": a_, "b": b_, "diff": m, "ic_bas": lo, "ic_haut": hi, "verdict": sig})
    pd.DataFrame(diffs).to_csv(RES / f"differences_bootstrap{sfx}.csv", index=False)

    # par type de question
    by_type = ans.groupby(["systeme", "type"]).correct_juge.mean().unstack().reindex(SYSTEMS).round(2)
    by_type.to_csv(RES / f"correct_par_type{sfx}.csv")
    print("\nCorrect (juge) par type de question :\n", by_type.to_string())

    # taxonomie d'erreurs (systèmes avec retrieval)
    errs = []
    for s in SYSTEMS[1:]:
        r5 = ret[ret.systeme == s].set_index("id")["recall@5"]
        for _, q in g[g.systeme == s].iterrows():
            if q.type == "sans_reponse":
                cat = "ok" if q.abstention else "hallucination (aurait dû s'abstenir)"
            elif q.correct_juge == 1:
                cat = "ok"
            elif q.abstention:
                cat = "refus injustifié" if r5.get(q.id, 0) > 0 else "passage non retrouvé (abstention)"
            elif 0 < r5.get(q.id, 0) < 1:
                cat = "information répartie : un seul des passages retrouvé"
            elif r5.get(q.id, 0) == 0:
                cat = "passage non retrouvé"
            elif q.fidele_juge == 0:
                cat = "hallucination (info absente du contexte)"
            else:
                cat = "bon contexte, réponse incomplète/fausse"
            errs.append({"systeme": s, "categorie": cat})
    e = pd.DataFrame(errs).value_counts(["systeme", "categorie"]).unstack(0).fillna(0).astype(int)
    e.to_csv(RES / f"taxonomie_erreurs{sfx}.csv")
    print("\nTaxonomie d'erreurs (nombre de réponses) :\n", e.to_string())

    # figures
    FIG.mkdir(exist_ok=True)
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    cs = [bootstrap_ci(ans[ans.systeme == s].correct_juge.fillna(0)) for s in SYSTEMS]
    X = range(len(SYSTEMS))
    ax[0].bar(X, [c[0] for c in cs], yerr=[[c[0] - c[1] for c in cs], [c[2] - c[0] for c in cs]], capsize=4, color="#4c78a8")
    ax[0].set_xticks(X); ax[0].set_xticklabels([LABEL[s].replace(" ", "\n", 1) for s in SYSTEMS], fontsize=8)
    ax[0].set_ylim(0, 1); ax[0].set_title("Réponses correctes (juge)\nquestions avec réponse")
    ab = [bootstrap_ci(noans[noans.systeme == s].abstention.astype(float)) if len(noans[noans.systeme == s]) else (0, 0, 0) for s in SYSTEMS]
    ax[1].bar(X, [c[0] for c in ab], yerr=[[c[0] - c[1] for c in ab], [c[2] - c[0] for c in ab]], capsize=4, color="#e45756")
    ax[1].set_xticks(X); ax[1].set_xticklabels([LABEL[s].replace(" ", "\n", 1) for s in SYSTEMS], fontsize=8)
    ax[1].set_ylim(0, 1); ax[1].set_title("Abstention correcte\nquestions sans réponse")
    fig.suptitle("Résultats provisoires (jeu de test non relu à la main)" if "provisoire" in gen_path.name else "", fontsize=9)
    fig.tight_layout(); fig.savefig(FIG / f"resultats_generation{sfx}.png", dpi=150)

    r = pd.read_csv(RES / "retrieval_structural_512.csv")
    fig, ax = plt.subplots(figsize=(6, 3.5))
    w = 0.2
    for k, col in enumerate(["recall@1", "recall@3", "recall@5", "mrr"]):
        ax.bar(np.arange(len(r)) + k * w, r[col], w, label=col)
    ax.set_xticks(np.arange(len(r)) + 1.5 * w); ax.set_xticklabels(r.systeme, fontsize=7, rotation=10)
    ax.set_ylim(0, 1); ax.legend(fontsize=7); ax.set_title("Retrieval (structurel 512)")
    fig.tight_layout(); fig.savefig(FIG / "retrieval.png", dpi=150)
    print("\nfigures :", [p.name for p in FIG.glob("*.png")])


CONFIGS = [f"{s}_{n}" for s in ("structural", "fixed") for n in (256, 512, 1024)]
RET_SYS = ["S1_bm25", "S2_dense", "RRF", "S3_hybride_rerank"]


def retrieval() -> None:
    """Ablation du découpage (6 configurations x 4 systèmes) avec IC bootstrap, et différences appariées."""
    per = {}
    for c in CONFIGS:
        p = RES / f"retrieval_par_question_{c}.csv"
        if p.exists():
            per[c] = pd.read_csv(p)
    rows = []
    for c, d in per.items():
        for s in RET_SYS:
            x = d[d.systeme == s]
            m5, lo5, hi5 = bootstrap_ci(x["recall@5"])
            rows.append({"découpage": c, "système": s, "Recall@1": round(x["recall@1"].mean(), 3),
                         "Recall@5": fmt(m5, lo5, hi5), "MRR": round(x["mrr"].mean(), 3), "n": len(x)})
    tab = pd.DataFrame(rows)
    tab.to_csv(RES / "ablation_decoupage.csv", index=False)
    print(tab.to_string(index=False))

    diffs = []
    ref = per["structural_512"]
    for metric in ("recall@1", "recall@5", "mrr"):
        for a_, b_ in [("S3_hybride_rerank", "S1_bm25"), ("S3_hybride_rerank", "S2_dense"), ("S1_bm25", "S2_dense"), ("RRF", "S1_bm25")]:
            x = ref[ref.systeme == a_].set_index("id")[metric]
            y = ref[ref.systeme == b_].set_index("id")[metric].reindex(x.index)
            m, lo, hi = paired_diff(x, y)
            diffs.append({"comparaison": f"{a_} - {b_}", "métrique": metric, "diff": round(m, 3), "ic_bas": round(lo, 3),
                          "ic_haut": round(hi, 3), "verdict": "significatif" if lo > 0 or hi < 0 else "non significatif"})
    for s in RET_SYS:                                    # structurel contre fixe, à taille égale
        for size in (256, 512, 1024):
            if f"fixed_{size}" in per and f"structural_{size}" in per:
                x = per[f"structural_{size}"].query("systeme == @s").set_index("id")["recall@5"]
                y = per[f"fixed_{size}"].query("systeme == @s").set_index("id")["recall@5"].reindex(x.index)
                m, lo, hi = paired_diff(x, y)
                diffs.append({"comparaison": f"{s} : structurel - fixe ({size})", "métrique": "recall@5", "diff": round(m, 3),
                              "ic_bas": round(lo, 3), "ic_haut": round(hi, 3),
                              "verdict": "significatif" if lo > 0 or hi < 0 else "non significatif"})
    dd = pd.DataFrame(diffs)
    dd.to_csv(RES / "differences_retrieval.csv", index=False)
    print("\n", dd.to_string(index=False))


def annotate(gen_path: Path, n: int = 30) -> None:
    g = pd.read_csv(gen_path)
    g = g[g.systeme.isin(SYSTEMS[1:])]
    s = g.sample(n=min(n, len(g)), random_state=SEED)
    out = pd.DataFrame({"id": s.id, "systeme": s.systeme, "question": s.question,
                        "reference": s.reference.fillna("(aucune réponse dans le corpus : s'abstenir est correct)"),
                        "reponse_du_systeme": s.reponse,
                        "ta_note": "", "commentaire": ""})
    # le juge n'est volontairement PAS montré : on évite d'influencer l'annotateur
    out.to_csv(RES / "annotations_a_remplir.csv", index=False, sep=";", encoding="utf-8-sig")
    s[["id", "systeme", "correct_juge"]].to_csv(RES / "_notes_juge_cachees.csv", index=False)
    print(f"{len(out)} réponses -> results/annotations_a_remplir.csv  (ta_note : 1 = correcte, 0.5 = partielle, 0 = fausse ou abstention à tort)")


def kappa(annot: str = "annotations_a_remplir.csv", col: str = "ta_note") -> None:
    """Accord entre les notes d'un annotateur (humain ou autre) et le juge LLM."""
    h = pd.read_csv(RES / annot, sep=";", encoding="utf-8-sig")
    j = pd.read_csv(RES / "_notes_juge_cachees.csv")
    d = h.merge(j, on=["id", "systeme"])
    d["ta_note"] = pd.to_numeric(d[col].astype(str).str.replace(",", "."), errors="coerce")
    d = d.dropna(subset=["ta_note", "correct_juge"])
    print(f"{len(d)} réponses annotées")
    print("accord brut :", round((d.ta_note == d.correct_juge).mean(), 3))
    h3, j3 = (d.ta_note * 2).astype(int), (d.correct_juge * 2).astype(int)     # 0 / 0.5 / 1  ->  classes 0 / 1 / 2
    print("kappa (3 niveaux 0/0.5/1) :", round(cohen_kappa_score(h3, j3), 3))
    print("kappa pondéré quadratique :", round(cohen_kappa_score(h3, j3, weights="quadratic"), 3))
    print("kappa binaire (1 vs pas 1) :", round(cohen_kappa_score(d.ta_note == 1, d.correct_juge == 1), 3))
    print(pd.crosstab(d.ta_note, d.correct_juge, rownames=["humain"], colnames=["juge"]))


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["report", "annotate", "kappa", "retrieval"])
    ap.add_argument("--gen", default=str(RES / "generations_provisoire.csv"))
    ap.add_argument("--tag", default="", help="suffixe des fichiers de sortie du rapport")
    ap.add_argument("--annot", default="annotations_a_remplir.csv")
    ap.add_argument("--col", default="ta_note")
    a = ap.parse_args()
    {"report": lambda: report(Path(a.gen), a.tag), "annotate": lambda: annotate(Path(a.gen)),
     "kappa": lambda: kappa(a.annot, a.col), "retrieval": retrieval}[a.cmd]()


if __name__ == "__main__":
    main()

**`src/run_retrieval.py`** — Comparaison S1 / S2 / RRF / S3 en retrieval

In [ ]:
%%writefile src/run_retrieval.py
"""Compare S1 (BM25), S2 (dense), RRF et S3 (RRF + reranker) en retrieval ; sauvegarde les classements.

Sortie : results/retrieval_<config>.csv (une ligne par système) et
data/processed/rankings_<config>.json (top-10 par question et par système, réutilisé pour la génération).
Usage : python -m src.run_retrieval [--questions chemin.jsonl] [--config structural_512]
"""
import argparse
import json
from pathlib import Path

import pandas as pd

from src import retrieval_eval as R
from src.bm25 import BM25
from src.dense import DenseIndex, encode, free_gpu as free_dense
from src.hybrid import rrf, rerank, free_gpu as free_rerank

ROOT = Path(__file__).resolve().parents[1]
CAND = 30      # candidats envoyés au reranker


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--questions", default=str(ROOT / "data/testset/brouillon.jsonl"))
    ap.add_argument("--config", default="structural_512")
    a = ap.parse_args()
    qs = [json.loads(l) for l in Path(a.questions).read_text(encoding="utf-8").splitlines()]
    ch = R.load_chunks(a.config)
    bm = BM25([c["context"] + "\n" + c["text"] for c in ch])
    dn = DenseIndex(ch, a.config)
    Q = encode([q["question"] for q in qs])
    free_dense()

    rank = {s: {} for s in ("S1_bm25", "S2_dense", "RRF", "S3_hybride_rerank")}
    for i, q in enumerate(qs):
        b = [j for j, _ in bm.search(q["question"], CAND)]
        d = dn.search_emb(Q[i:i + 1], CAND)[0][0].tolist()
        f = rrf([b, d], top=CAND)
        rank["S1_bm25"][q["id"]] = b[:10]
        rank["S2_dense"][q["id"]] = d[:10]
        rank["RRF"][q["id"]] = f[:10]
        rank["S3_hybride_rerank"][q["id"]] = rerank(q["question"], ch, f, k=10)
    free_rerank()

    (ROOT / "data/processed").mkdir(exist_ok=True)
    (ROOT / f"data/processed/rankings_{a.config}.json").write_text(json.dumps(rank), encoding="utf-8")
    rows, per_q = [], []
    for name, rk in rank.items():
        res = R.evaluate(lambda q, k, rk=rk, qmap={q["question"]: q["id"] for q in qs}: rk[qmap[q]][:k], ch, qs)
        rows.append({"systeme": name, **R.summarize(res)})
        per_q += [{"systeme": name, **r} for r in res["per_question"]]
    (ROOT / "results").mkdir(exist_ok=True)
    out = pd.DataFrame(rows)
    out.to_csv(ROOT / f"results/retrieval_{a.config}.csv", index=False)
    pd.DataFrame(per_q).to_csv(ROOT / f"results/retrieval_par_question_{a.config}.csv", index=False)
    print(out.round(3).to_string(index=False))


if __name__ == "__main__":
    main()

**`src/run_generation.py`** — Génération et notation S0..S3

In [ ]:
%%writefile src/run_generation.py
"""Génère et note les réponses pour S0..S3 sur un jeu de questions. Reprise automatique (cache LLM).

Usage : python -m src.run_generation --questions data/testset/questions.jsonl --n 60 --systems S0 S1_bm25 S2_dense S3_hybride_rerank
Sortie : results/generations.csv (une ligne par question et par système)
Les classements doivent exister : python -m src.run_retrieval --questions <même fichier>
"""
import argparse
import json
import random
from pathlib import Path

import pandas as pd

from src import rag
from src import retrieval_eval as R

ROOT = Path(__file__).resolve().parents[1]


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--questions", default=str(ROOT / "data/testset/questions.jsonl"))
    ap.add_argument("--config", default="structural_512")
    ap.add_argument("--systems", nargs="+", default=["S0", "S1_bm25", "S2_dense", "S3_hybride_rerank"])
    ap.add_argument("--k", type=int, default=5)
    ap.add_argument("--n", type=int, default=0, help="sous-échantillon stratifié par type (0 = tout)")
    ap.add_argument("--subset", type=int, default=0, help="sous-échantillon stratifié du jeu déjà tiré avec --n")
    ap.add_argument("--prompt", choices=["cite", "simple"], default="cite")
    ap.add_argument("--out", default=str(ROOT / "results/generations.csv"))
    a = ap.parse_args()

    qs = [json.loads(l) for l in Path(a.questions).read_text(encoding="utf-8").splitlines()]

    def stratified(items, n, seed):
        rng, by = random.Random(seed), {}
        for q in items:
            by.setdefault(q["type"], []).append(q)
        return [q for v in by.values() for q in rng.sample(v, max(1, round(n * len(v) / len(items))))]

    if a.n:
        qs = stratified(qs, a.n, 42)
    if a.subset:                                         # sous-échantillon du précédent (les réponses déjà en cache sont réutilisées)
        qs = stratified(qs, a.subset, 7)
    ch = R.load_chunks(a.config)
    rank = json.loads((ROOT / f"data/processed/rankings_{a.config}.json").read_text(encoding="utf-8"))
    prompt = rag.PROMPT_RAG_CITE if a.prompt == "cite" else rag.PROMPT_RAG_SIMPLE

    rows = []
    out = Path(a.out)
    try:
        run(a, qs, ch, rank, prompt, rows, out)
    except RuntimeError as e:                            # quota journalier atteint : on garde ce qui est fait
        pd.DataFrame(rows).to_csv(out, index=False)
        print(f"ARRÊT (quota) après {len(rows)} lignes sauvegardées : {str(e)[:160]}")
        raise SystemExit(2)


def run(a, qs, ch, rank, prompt, rows, out) -> None:
    for s in a.systems:
        for n, q in enumerate(qs, 1):
            chunks = None if s == "S0" else [ch[i] for i in rank[s][q["id"]][:a.k]]
            g = rag.answer(q["question"], chunks, prompt)
            ref = q["reponse"]
            j = rag.judge(q["question"], ref or "(aucune réponse dans le corpus : l'abstention est la bonne réponse)",
                          g["text"], chunks)
            abst = rag.is_abstention(g["text"])
            rows.append({"systeme": s, "id": q["id"], "type": q["type"], "question": q["question"],
                         "reference": ref, "reponse": g["text"], "abstention": abst,
                         "f1": rag.f1(g["text"], ref) if ref else None,
                         "correct_juge": (1.0 if abst else 0.0) if ref is None else j["correct"],
                         "fidele_juge": j["fidele"], "raison_juge": j["raison"],
                         "citation_ok": rag.citation_ok(g["text"], len(chunks)) if chunks else None,
                         "prompt_tokens": g["prompt_tokens"], "completion_tokens": g["completion_tokens"],
                         "latency_s": g["latency_s"], "judge_tokens": j["judge_tokens"]})
            if n % 10 == 0:
                print(f"{s}: {n}/{len(qs)}", flush=True)
                pd.DataFrame(rows).to_csv(out, index=False)
    pd.DataFrame(rows).to_csv(out, index=False)
    print("terminé ->", out)


if __name__ == "__main__":
    main()

**`src/download_corpus.py`** — Téléchargement du corpus

In [ ]:
%%writefile src/download_corpus.py
"""Téléchargement automatique du corpus (brochures INRS + textes EUR-Lex).

Les PDF vont dans data/raw/ (ignoré par git). La liste des sources
(URL, titre, nb de pages) est écrite dans data/sources.csv, qui lui est suivi
par git : c'est la « source du dataset » demandée au notebook.

Usage : python -m src.download_corpus
"""
import csv
import time
from pathlib import Path

import pymupdf
import requests

ROOT = Path(__file__).resolve().parents[1]
RAW = ROOT / "data" / "raw"
SOURCES_CSV = ROOT / "data" / "sources.csv"

HEADERS = {"User-Agent": "Mozilla/5.0 (projet-RAG-securite-machines)"}
INRS_URL = "https://www.inrs.fr/dam/inrs/CataloguePapier/ED/TI-ED-{n}.pdf"
EURLEX_URL = "https://eur-lex.europa.eu/legal-content/FR/TXT/PDF/?uri=CELEX:{celex}"

# Brochures INRS candidates, repérées sur les pages « publications » INRS
# consacrées à la conception et à l'utilisation des machines.
INRS_REFS = [
    54, 113, 114, 123, 129, 134, 781, 828, 879, 912, 4450, 5100, 6016, 6038,
    6104, 6108, 6109, 6112, 6122, 6129, 6154, 6176, 6178, 6270, 6281, 6310,
    6339, 6389,
    # Ajoutées via le catalogue INRS « Machines » (ED 4707) et des recherches ciblées.
    126, 6231, 6289, 6323, 6330, 6386, 6446, 6499, 6531, 6540,
]

# Titres absents ou mal extraits des métadonnées PDF.
TITLE_OVERRIDES = {
    "INRS_ED6038.pdf": "Intervention sur un équipement de travail. Réflexions pour la sécurité",
    "EURLEX_directive_2006_42_CE_machines.pdf": "Directive 2006/42/CE du Parlement européen et du Conseil du 17 mai 2006 relative aux machines",
    "EURLEX_reglement_2023_1230_machines.pdf": "Règlement (UE) 2023/1230 du Parlement européen et du Conseil du 14 juin 2023 sur les machines",
}

# Brochures dont l'URL PDF ne suit pas le schéma TI-ED-{n}.pdf.
INRS_URL_OVERRIDES = {
    6499: "https://www.inrs.fr/dam/jcr:cd912671-9252-4bc0-a941-21efbc69ab58/ed6499.pdf",
    6330: "https://www.inrs.fr/dms/inrs/CataloguePapier/ED/TI-ED-6330/ed6330.pdf",
}

# (nom de fichier, CELEX)
EURLEX_DOCS = [
    ("EURLEX_reglement_2023_1230_machines", "32023R1230"),
    ("EURLEX_directive_2006_42_CE_machines", "32006L0042"),
]


def fetch_pdf(url: str, dest: Path) -> bool:
    """Télécharge url vers dest ; renvoie True seulement si c'est un vrai PDF."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=60, allow_redirects=True)
    except requests.RequestException as e:
        print(f"  ! erreur réseau : {e}")
        return False
    if r.status_code != 200 or not r.content.startswith(b"%PDF"):
        print(f"  ! HTTP {r.status_code}, pas un PDF")
        return False
    dest.write_bytes(r.content)
    return True


def describe(path: Path) -> tuple[str, int]:
    """Titre (métadonnées, sinon début de la 1re page) et nombre de pages."""
    with pymupdf.open(path) as doc:
        title = (doc.metadata or {}).get("title", "").strip()
        if not title and doc.page_count:
            title = " ".join(doc[0].get_text().split())[:120]
        return title, doc.page_count


def main() -> None:
    RAW.mkdir(parents=True, exist_ok=True)
    jobs = [(f"INRS_ED{n}", INRS_URL_OVERRIDES.get(n, INRS_URL.format(n=n))) for n in INRS_REFS]
    jobs += [(name, EURLEX_URL.format(celex=c)) for name, c in EURLEX_DOCS]

    rows, failed = [], []
    for name, url in jobs:
        dest = RAW / f"{name}.pdf"
        print(f"{name}")
        if dest.exists() or fetch_pdf(url, dest):
            title, pages = describe(dest)
            title = TITLE_OVERRIDES.get(dest.name, title)
            rows.append({"fichier": dest.name, "url": url, "titre": title, "pages": pages})
            print(f"  OK  {pages} p. | {title[:80]}")
        else:
            failed.append(name)
        time.sleep(1)  # politesse envers les serveurs

    with open(SOURCES_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["fichier", "url", "titre", "pages"])
        w.writeheader()
        w.writerows(rows)

    print(f"\n{len(rows)} documents OK, {len(failed)} échecs : {failed}")


if __name__ == "__main__":
    main()

## 6. Protocole expérimental

- **Retrieval** : un chunk est *pertinent* si, dans le bon document, il couvre au moins 60 % des triplets de mots de la preuve. Cette définition ne dépend pas du découpage, ce qui permet de comparer toutes les configurations. **Recall@k** (part des preuves couvertes par les k premiers chunks) et **MRR** (inverse du rang du premier bon chunk). Les questions sans réponse sont exclues du retrieval. Évalué sur les **132 questions avec preuve**, pour les **6 configurations de découpage** et les 4 systèmes de recherche.
- **Génération** (k = 5 chunks, température 0) : F1 sur les tokens, correction et fidélité par le juge, **abstention correcte** sur les questions sans réponse, **refus injustifié** sur les autres, exactitude formelle des citations, tokens et latence. Deux échantillons stratifiés par type de question : **61 questions** pour S0 et S1, et un sous-échantillon de **34 questions** sur lequel les **4 systèmes** sont comparés (le quota gratuit de l'API impose cette taille).
- **Validation du juge** : 30 réponses annotées. **Les annotations de ce notebook sont de Claude, pas d'un humain** : le kappa mesure donc l'accord entre deux LLM, ce qui est une validation plus faible qu'un accord humain–juge.
- **Jeu de test** : 20 questions factuelles ont été **relues et validées à la main** par l'auteur (§7.5) ; les autres sont vérifiées automatiquement (preuve retrouvée dans le document) mais non relues.
- **Significativité** : bootstrap (5 000 rééchantillonnages, graine 42), avec **différences appariées** sur les mêmes questions : une différence est dite significative si son intervalle à 95 % exclut 0.

## 7. Résultats

### 7.1 Retrieval (structurel 512, 132 questions)

In [ ]:
from IPython.display import Image, display
r = pd.read_csv(ROOT / "results/retrieval_structural_512.csv").round(3)
display(r)
d = pd.read_csv(ROOT / "results/differences_retrieval.csv")
display(d[~d.comparaison.str.contains("structurel")])
Image(ROOT / "figures/retrieval.png")

- **S3 (hybride + reranker) est le meilleur** : par rapport à BM25, **Recall@1 +0,10 [0,03 ; 0,17]** et **MRR +0,07 [0,02 ; 0,12]** (significatifs). En revanche **Recall@5 +0,03 [0,00 ; 0,07] n'est pas significatif** : à 5 passages, BM25 retrouve déjà presque tout, et le reranker améliore surtout le *classement* (le bon passage arrive plus souvent en tête).
- **La fusion RRF seule n'améliore pas BM25** (écart non significatif, légèrement négatif) : fusionner un bon classement avec un moins bon le dilue.
- **BM25 bat le dense**, significativement (Recall@1 +0,19, MRR +0,14, Recall@5 +0,07). Les questions étant générées par un LLM à partir du texte, elles en reprennent souvent les mots : ce jeu de test **avantage la recherche lexicale** (§9).

### 7.2 Ablation : stratégie et taille des chunks (Recall@5 avec IC à 95 %)

In [ ]:
ab = pd.read_csv(ROOT / "results/ablation_decoupage.csv")
display(ab.pivot(index="découpage", columns="système", values="Recall@5")[["S1_bm25", "S2_dense", "RRF", "S3_hybride_rerank"]])
print("Différences appariées structurel - fixe (Recall@5) :")
display(d[d.comparaison.str.contains("structurel")][["comparaison", "diff", "ic_bas", "ic_haut", "verdict"]].reset_index(drop=True))

- **Le découpage structurel est meilleur que le fixe, mais pas partout.** L'avantage est **significatif à 512 tokens** pour BM25 (+0,05), le dense (+0,11) et S3 (+0,03), et pour le dense à 1024 tokens (+0,08). Il **n'est pas significatif à 256 tokens**, ni pour la fusion RRF.
- **Taille** : 512 tokens est la meilleure configuration pour BM25 (0,92) et pour S3 (0,96) ; 256 tokens est un peu moins bon, 1024 est comparable. Les intervalles se recouvrent : l'effet de la taille est **modeste**.
- Le dense profite le plus de la structure : un chunk de fixe qui coupe un article en deux dilue le sens que l'embedding doit capturer.

### 7.3 Génération : les 4 systèmes sur 34 questions

In [ ]:
g4 = pd.read_csv(ROOT / "results/generations_35.csv")
print(g4.groupby("systeme").size().to_string(), "\n")
display(pd.read_csv(ROOT / "results/metriques_par_systeme_4sys.csv"))
display(pd.read_csv(ROOT / "results/differences_bootstrap_4sys.csv").round(3))
display(pd.read_csv(ROOT / "results/correct_par_type_4sys.csv", index_col=0))
display(pd.read_csv(ROOT / "results/taxonomie_erreurs_4sys.csv", index_col=0))
Image(ROOT / "figures/resultats_generation_4sys.png")

Résultats sur **34 questions** (27 avec réponse, 7 sans), les mêmes pour les quatre systèmes :
- **Le retrieval est ce qui compte.** Réponses correctes : S0 **0,28** [0,15 ; 0,41], S1 0,81, S2 0,89, **S3 0,89** [0,81 ; 0,96]. Différence S3 − S0 : **+0,61 [0,43 ; 0,78]**, significative.
- **Entre systèmes RAG, aucune différence significative** : S3 − S1 = +0,07 [−0,02 ; 0,20], S3 − S2 = 0,00. Le gain de S3 observé en retrieval (Recall@1, MRR) **ne se traduit pas ici en gain mesurable de génération** : avec 5 extraits, le LLM retrouve l'information même si le bon passage n'est pas en tête, et 27 questions donnent trop peu de puissance statistique pour détecter un écart de 7 points.
- **Abstention** (7 questions seulement) : S0 s'abstient **0 fois sur 7** ; S1 et S2 4 sur 7, S3 5 sur 7. Intervalles très larges (par ex. [0,29 ; 1,00] pour S3) : on ne peut **pas** classer les systèmes RAG sur ce critère. Le RAG ne supprime pas l'hallucination : 2 à 3 questions sans réponse sur 7 reçoivent encore une réponse inventée.
- **Fidélité et citations** : 0,97 à 1,00 de fidélité aux extraits ; toutes les réponses non abstentives contiennent des citations, **toutes avec des numéros d'extraits valides** (validité *formelle* : on ne vérifie pas que l'extrait cité soutient bien l'affirmation).
- **Multi-passages** : les plus difficiles (S1 0,40 ; S2 0,60 ; S3 0,80), mais sur très peu de questions : indicatif.
- **Erreurs (sur 34)** : S1 — 3 hallucinations, 3 refus injustifiés, 2 « bon contexte mais réponse incomplète ou fausse », 2 passages non retrouvés ; S2 — 3, 1, 2, 1 et 1 « information répartie » ; S3 — 2 hallucinations, 1 refus injustifié, 4 « bon contexte mais réponse incomplète ou fausse », 1 passage non retrouvé. Avec S3, le retrieval échoue moins, mais le **goulot se déplace vers la génération** (réponses incomplètes malgré un bon contexte).
- **Coût** : environ 2 100 à 2 250 tokens par requête RAG contre 260 sans documents (×8) ; latence ~0,6 s. Le coût en dollars de la table est **indicatif** (tarifs non vérifiés).

#### S0 contre S1 sur 61 questions

In [ ]:
display(pd.read_csv(ROOT / "results/metriques_par_systeme_2sys.csv"))
Image(ROOT / "figures/resultats_generation_2sys.png")

Sur un échantillon plus large (**61 questions, 48 avec réponse**), le constat est le même : réponses correctes **24 % → 78 %** avec BM25 (différence +0,54 [0,41 ; 0,68]), abstentions correctes 15 % → 77 % (13 questions), fidélité 0,98. Multi-passages : 50 % avec BM25.

### 7.4 Validité du juge

In [ ]:
from sklearn.metrics import cohen_kappa_score
a = pd.read_csv(ROOT / "results/annotations_claude.csv", sep=";", encoding="utf-8-sig")
j = pd.read_csv(ROOT / "results/_notes_juge_cachees.csv")
d_ = a.merge(j, on=["id", "systeme"])
h3, j3 = (d_.claude_note * 2).astype(int), (d_.correct_juge * 2).astype(int)
print(f"{len(d_)} réponses | accord brut {np.mean(h3 == j3):.3f} | kappa {cohen_kappa_score(h3, j3):.3f} | kappa pondéré {cohen_kappa_score(h3, j3, weights='quadratic'):.3f}")
pd.crosstab(d_.claude_note, d_.correct_juge, rownames=["annotation (Claude)"], colnames=["juge"])

Accord élevé (29/30), mais c'est un accord **entre deux LLM**, sur un échantillon facile (29 réponses de S1 sur 30), jugé par rapport à la référence seule : à ne pas surinterpréter.

### 7.5 Questions relues à la main

In [ ]:
rel = [json.loads(l) for l in open(ROOT / "data/testset/questions.jsonl", encoding="utf-8")]
print(len(rel), "questions relues et validées :", pd.Series([q["type"] for q in rel]).value_counts().to_dict())
pd.read_csv(ROOT / "results/retrieval_questions_relues.csv")

**20 questions factuelles** ont été relues et validées à la main par l'auteur (toutes conservées). Sur ce petit jeu validé, le **classement des systèmes de retrieval est le même** que sur les 132 questions : S3 (Recall@1 0,80) devant BM25 (0,55), RRF (0,50) et le dense (0,40). Les intervalles sont larges (n = 20). Ces 20 questions ne contiennent **aucune** des 34 questions de la comparaison à 4 systèmes : la génération n'a donc pas été évaluée sur des questions relues.

## 8. Interprétation et analyse critique

**1. Le RAG est indispensable.** Sans documents, le LLM répond correctement à environ un quart des questions (24 à 28 %) et ne s'abstient jamais ou presque quand la réponse n'existe pas : il invente des valeurs précises et plausibles (ex. une « distance de sécurité de 0,5 m si la vitesse est ≤ 0,5 m/s », une « inspection annuelle »). Avec des extraits, il passe à 78–89 % de réponses correctes, avec des affirmations soutenues par les extraits dans 97 à 100 % des cas. C'est le résultat le plus solide (+0,54 à +0,61, significatif sur les deux échantillons) : sur un domaine où une valeur exacte compte, l'apport du retrieval est massif.

**2. Le reranker améliore le classement, pas la couverture.** Il gagne +0,10 en Recall@1 et +0,07 en MRR sur BM25 (significatif), mais le gain en Recall@5 (+0,03) ne l'est pas : à cinq passages, BM25 couvre déjà presque tout. La fusion RRF seule n'apporte rien (elle dilue le meilleur classement), donc **le gain de S3 vient de la relecture fine du couple question-passage**, pas de la combinaison lexical + dense.

**3. Un meilleur retrieval ne donne pas forcément une meilleure génération.** En génération (34 questions), S3 ≈ S2 > S1 sans différence significative. Hypothèses : le générateur retrouve l'information dans les 5 extraits quelle que soit leur position ; l'échantillon est trop petit pour détecter un écart de 7 points ; le jeu de test, qui reprend le vocabulaire du texte, rend BM25 déjà très bon. Ce résultat nuance l'intérêt de S3 : **plus coûteux et plus complexe, pour un gain mesuré en retrieval mais pas démontré en génération**.

**4. Le découpage structurel aide, avec nuance.** Il est significativement meilleur à 512 tokens (BM25, dense, S3) et pour le dense à 1024, mais pas à 256 tokens : l'effet est réel mais pas général, et la taille des chunks compte peu (512 légèrement devant).

**5. BM25 bat le dense** sur ce corpus (significatif). Explication probable : vocabulaire très technique (sigles, normes, valeurs) et questions qui reprennent les mots du texte. Ce n'est pas un jugement général sur les embeddings, mais une propriété de ce jeu de test.

**6. Les hallucinations ne disparaissent pas avec le RAG.** 2 à 3 questions sans réponse sur 7 reçoivent encore une réponse inventée (le prompt impose pourtant l'abstention), et les refus injustifiés existent (4 à 11 % des questions avec réponse). Avec S3 l'erreur dominante devient « bon contexte mais réponse incomplète ou fausse » : le **goulot passe du retrieval à la génération**.

**7. Les questions multi-passages sont les plus difficiles** (S1 40 à 50 %, S3 80 % sur quelques questions) : il faut retrouver et combiner deux passages.

**Coût.** Un appel RAG consomme environ 2 100 à 2 250 tokens contre 260 sans documents (×8), pour une latence de ~0,6 s. La qualité se paie en tokens, et c'est ce qui a limité l'échelle de l'expérience avec un quota gratuit.

**Esprit critique.** Les écarts très grands (S0 contre RAG) sont robustes aux défauts de l'étude. Les écarts fins (S3 contre S1 ou S2 en génération, comparaison des systèmes RAG sur l'abstention avec 7 questions) ne sont **pas démontrés** : intervalles larges, juge validé seulement par un autre LLM, jeu de test en grande partie non relu.

## 9. Limites et perspectives

**Limites qui conditionnent la lecture des résultats**
- **Jeu de test** : seules **20 questions factuelles** ont été relues à la main. Les autres ont leur preuve vérifiée automatiquement mais questions et références ne sont pas relues ; quelques références sont imprécises. La génération n'a pas été évaluée sur des questions relues.
- **Biais du jeu de test** : questions générées par un LLM à partir de chunks structurels de 512 tokens, qui reprennent le vocabulaire du texte (avantage probable pour BM25 et pour cette configuration).
- **Petits échantillons en génération** : 34 questions (27 avec réponse, 7 sans) pour les 4 systèmes, 61 pour S0 et S1 ; les intervalles de confiance sont larges, surtout pour l'abstention.
- **Validation du juge** : les 30 annotations ont été produites par Claude, non par un humain ; le kappa mesure un accord entre deux LLM. L'exactitude des citations n'est vérifiée que formellement.
- **Corpus** : dominé par l'INRS (38 documents sur 40) ; figures non extraites ; tableaux sur plusieurs pages parfois mal reconstitués.
- **Choix de modèles** : un seul générateur, un seul juge, un seul encodeur et un seul reranker ; coûts en dollars indicatifs.

**Perspectives**
- Agrandir l'échantillon de génération pour départager les systèmes RAG (le quota gratuit de l'API est le facteur limitant ; le code reprend grâce au cache).
- Relire tout le jeu de test et mesurer un kappa humain–juge.
- Ablations de génération : top-k, prompt simple contre prompt à citations, HyDE, query rewriting, petit contre grand LLM.
- Fine-tuning léger des embeddings sur le vocabulaire du domaine ; essai d'autres encodeurs et rerankers.

## 10. Conclusion

Sur un corpus de 40 documents français sur la sécurité des machines, un pipeline RAG codé à la main montre que :
- **le retrieval réduit fortement les hallucinations** : réponses correctes 24–28 % sans documents contre 78–89 % avec, et fidélité aux extraits de 97 à 100 % ; l'écart (+0,54 à +0,61) est significatif ;
- **l'hybride avec reranker est le meilleur système de retrieval** (Recall@1 0,82 contre 0,72 pour BM25 et 0,53 pour le dense ; MRR 0,93 contre 0,86 et 0,72), mais **ce gain ne se confirme pas en génération** sur 34 questions (+0,07, non significatif) ;
- le **découpage structurel** améliore significativement la recherche à 512 tokens, sans être un gain général ;
- le RAG **ne supprime pas** les hallucinations sur les questions hors corpus (2 à 3 sur 7) et déplace l'erreur dominante de la recherche vers la génération.

Ces résultats sont **solides sur les grands écarts** (RAG contre LLM seul) et **préliminaires sur les écarts fins**, faute d'un jeu de test intégralement relu et d'échantillons de génération plus larges.

## 11. Références et source du dataset

**Dataset** : brochures INRS (inrs.fr), Règlement (UE) 2023/1230 et Directive 2006/42/CE (EUR-Lex). Liste complète, avec l'URL de chaque document, dans `data/sources.csv` (reproduite ci-dessous).

**Code et résultats** : https://github.com/Mohamed-Ghandour/rag-securite-machines

In [ ]:
pd.set_option("display.max_colwidth", 110)
pd.read_csv(ROOT / "data/sources.csv")[["fichier", "titre", "pages", "url"]]

**Documents consultables** : les PDF ne sont pas redistribués (droits des éditeurs) ; `python -m src.download_corpus` les retélécharge depuis les URL ci-dessus.

**Références**
- Lewis et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS.
- Robertson & Zaragoza (2009). *The Probabilistic Relevance Framework: BM25 and Beyond*.
- Chen et al. (2024). *BGE M3-Embedding: Multi-Lingual, Multi-Functionality, Multi-Granularity Text Embeddings*.
- Cormack, Clarke & Büttcher (2009). *Reciprocal Rank Fusion outperforms Condorcet and individual rank learning methods*. SIGIR.
- Zheng et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena*. NeurIPS.
- Cohen (1960). *A coefficient of agreement for nominal scales*.